In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Install R
!apt-get install -y libcurl4-openssl-dev libssl-dev libxml2-dev
%load_ext rpy2.ipython

In [ ]:
!apt-get update
!apt-get install -y build-essential libcurl4-openssl-dev libssl-dev libxml2-dev


In [ ]:
%%R
# Install
if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager")
}
if (!requireNamespace("devtools", quietly = TRUE)) {
  install.packages("devtools")
}
if (!requireNamespace("Seurat", quietly = TRUE)) {
  install.packages("Seurat")
}
if (!requireNamespace("Matrix", quietly = TRUE)) {
  install.packages("Matrix")
}
if (!requireNamespace("dplyr", quietly = TRUE)) {
  install.packages("dplyr")
}
if (!requireNamespace("patchwork", quietly = TRUE)) {
  install.packages("patchwork")
}

# Install Bioconductor packages
if (!requireNamespace("SingleR", quietly = TRUE)) {
  BiocManager::install("SingleR")
}
if (!requireNamespace("celldex", quietly = TRUE)) {
  BiocManager::install("celldex")
}




In [ ]:
%%R
install.packages(c("Rcpp", "RcppArmadillo", "devtools", "remotes"))


In [ ]:
%%R
# Install BayesPrism dependencies
if (!requireNamespace("snowfall", quietly = TRUE)) {
  install.packages("snowfall")
}
if (!requireNamespace("NMF", quietly = TRUE)) {
  install.packages("NMF")
}
if (!requireNamespace("scran", quietly = TRUE)) {
  BiocManager::install("scran")
}
if (!requireNamespace("gplots", quietly = TRUE)) {
  install.packages("gplots")
}

In [ ]:
%%R
if (!requireNamespace("devtools", quietly = TRUE)) {
    install.packages("devtools")
}

devtools::install_github("Danko-Lab/BayesPrism",
                         subdir = "BayesPrism",  # This was missing!
                         dependencies = FALSE,
                         force = TRUE)

In [ ]:
# System libs Seurat/igraph/hdf5/cairo
!sudo apt-get update
!sudo apt-get install -y \
  build-essential gfortran \
  libssl-dev libcurl4-openssl-dev libxml2-dev \
  libhdf5-dev libglpk-dev libgmp3-dev \
  libcairo2-dev libxt-dev \
  libfreetype6-dev libfontconfig1-dev libharfbuzz-dev libfribidi-dev \
  libpng-dev libtiff5-dev libjpeg-dev


In [ ]:
%%R
# Install
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")


BiocManager::install(ask = FALSE, update = TRUE)


BiocManager::install(c(
  "BiocGenerics", "S4Vectors", "IRanges", "MatrixGenerics", "BiocParallel"
), ask = FALSE, update = TRUE)


BiocManager::install(c("ROCR", "Seurat"),
                     ask = FALSE, update = TRUE)


library(ROCR);   packageVersion("ROCR")
library(Seurat); packageVersion("Seurat")


In [ ]:
%%R
# Load required libraries
library(Seurat)
library(Matrix)
library(dplyr)
library(patchwork)
library(SingleR)
library(celldex)
library(BayesPrism)


In [ ]:
%%R
scRNA_dir <- "/content/drive/MyDrive/BayesPrism/scRNA_data"

data <- Read10X(data.dir = scRNA_dir)
seurat_obj <- CreateSeuratObject(counts = data, project = "GSM7502530", min.cells = 3, min.features = 200)

# Robust mitochondrial and ribosomal gene detection
rn <- rownames(seurat_obj)
mt_pat <- if (any(grepl("^MT-", rn))) "^MT-" else if (any(grepl("^mt-", rn))) "^mt-" else "^MT-"
seurat_obj[["percent.mt"]] <- PercentageFeatureSet(seurat_obj, pattern = mt_pat)
seurat_obj[["percent.ribo"]] <- PercentageFeatureSet(seurat_obj, pattern = "^RP[SL]")

# Initial QC filtering with data-aware gates
feats <- seurat_obj$nFeature_RNA
counts <- seurat_obj$nCount_RNA
mt_pct <- seurat_obj$percent.mt

# Dynamic gates based on data distribution
feat_lo <- max(200, quantile(feats, 0.01, na.rm = TRUE))
feat_hi <- min(6000, quantile(feats, 0.99, na.rm = TRUE))
count_hi <- quantile(counts, 0.98, na.rm = TRUE)
mt_thr <- min(20, quantile(mt_pct, 0.95, na.rm = TRUE))

seurat_obj <- subset(seurat_obj, subset = nFeature_RNA > feat_lo &
                                        nFeature_RNA < feat_hi &
                                        nCount_RNA < count_hi &
                                        percent.mt < mt_thr)

In [ ]:
%%R
# Enhanced immune cell identification and filtering
expr_data <- GetAssayData(seurat_obj, slot = "counts")

# Convert sparse matrix to dense for easier processing
expr_data <- as.matrix(expr_data)

# PTPRC-based immune gating using raw counts
if("PTPRC" %in% rownames(expr_data)) {
  ptprc_expr <- as.numeric(expr_data["PTPRC", ])
  # Use a more appropriate threshold for raw counts
  ptprc_threshold <- quantile(ptprc_expr[ptprc_expr > 0], 0.25, na.rm = TRUE)
  if(is.na(ptprc_threshold)) ptprc_threshold <- 1

  # Additional immune markers for robust gating
  immune_markers <- c("PTPRC", "CD3E", "CD19", "CD14", "FCGR3A", "CD68", "MS4A1")
  available_immune <- intersect(immune_markers, rownames(expr_data))

  if(length(available_immune) > 1) {
    # For raw counts, check for any expression above 0
    immune_matrix <- expr_data[available_immune, , drop = FALSE]
    immune_score <- colSums(immune_matrix > 0) / length(available_immune)
    immune_threshold <- 0.2  # At least 20% of immune markers expressed
  } else {
    immune_score <- as.numeric(ptprc_expr > 0)
    immune_threshold <- 0.5
  }

  # Filter for epithelial and stromal contamination
  epithelial_markers <- c("EPCAM", "KRT8", "KRT18", "KRT19", "KRT7", "TACSTD2")
  stromal_markers <- c("COL1A1", "COL1A2", "COL3A1", "DCN", "BGN", "SPARC")

  available_epi <- intersect(epithelial_markers, rownames(expr_data))
  available_str <- intersect(stromal_markers, rownames(expr_data))

  epi_score <- if(length(available_epi) > 0) {
    epi_matrix <- expr_data[available_epi, , drop = FALSE]
    colSums(epi_matrix > 0) / length(available_epi)
  } else rep(0, ncol(expr_data))

  str_score <- if(length(available_str) > 0) {
    str_matrix <- expr_data[available_str, , drop = FALSE]
    colSums(str_matrix > 0) / length(available_str)
  } else rep(0, ncol(expr_data))

  # Define thresholds for contamination
  epi_threshold <- 0.5  # High epithelial marker expression
  str_threshold <- 0.5  # High stromal marker expression

  # Define immune cells: high immune markers, low epithelial/stromal
  is_immune <- (ptprc_expr >= ptprc_threshold | immune_score >= immune_threshold) &
               epi_score < epi_threshold &
               str_score < str_threshold

  seurat_obj$is_immune <- is_immune
} else {
  # Fallback: use other immune markers if PTPRC not found
  immune_markers <- c("CD3E", "CD19", "CD14", "FCGR3A", "CD68", "MS4A1")
  available_immune <- intersect(immune_markers, rownames(expr_data))

  if(length(available_immune) > 0) {
    immune_matrix <- expr_data[available_immune, , drop = FALSE]
    immune_score <- colSums(immune_matrix > 0) / length(available_immune)
    seurat_obj$is_immune <- immune_score >= 0.2
  } else {
    seurat_obj$is_immune <- TRUE
  }
}

# Subset to immune cells only
cd45_cells <- subset(seurat_obj, subset = is_immune == TRUE)


In [ ]:
%%R

# Re-run preprocessing on immune-only subset
set.seed(42)
cd45_cells <- NormalizeData(cd45_cells)
cd45_cells <- FindVariableFeatures(cd45_cells, selection.method = "vst", nfeatures = 2000)
cd45_cells <- ScaleData(cd45_cells, vars.to.regress = c("percent.mt", "nCount_RNA"))
cd45_cells <- RunPCA(cd45_cells)

set.seed(42)
cd45_cells <- RunUMAP(cd45_cells, dims = 1:30)
cd45_cells <- FindNeighbors(cd45_cells, dims = 1:30)
cd45_cells <- FindClusters(cd45_cells, resolution = 0.5)

In [ ]:
%%R
references <- list(
  DICE = try(celldex::DatabaseImmuneCellExpressionData()),
  Monaco = try(celldex::MonacoImmuneData()),
  Novershtern = try(celldex::NovershternHematopoieticData()),
  Blueprint = try(celldex::BlueprintEncodeData())
)
references <- references[!sapply(references, inherits, "try-error")]

cd45_data <- GetAssayData(cd45_cells, layer = "data")
for (ref_name in names(references)) {
  ref <- references[[ref_name]]
  singler_main <- SingleR(test = cd45_data, ref = ref, labels = ref$label.main)
  singler_fine <- SingleR(test = cd45_data, ref = ref, labels = ref$label.fine)
  cd45_cells[[paste0("cell_type_", tolower(ref_name), "_main")]] <- singler_main$labels
  cd45_cells[[paste0("cell_type_", tolower(ref_name), "_fine")]] <- singler_fine$labels
  saveRDS(singler_main, paste0("singler_", tolower(ref_name), "_main.rds"))
  saveRDS(singler_fine, paste0("singler_", tolower(ref_name), "_fine.rds"))
}

In [ ]:
%%R
best_ref <- ifelse("monaco" %in% tolower(names(references)), "monaco", "dice")

cell_type_col <- paste0("cell_type_", best_ref, "_main")
if (!cell_type_col %in% names(cd45_cells@meta.data)) {
  stop(paste("Column", cell_type_col, "not found in cd45_cells metadata"))
}

cell_types <- cd45_cells@meta.data[[cell_type_col]]
cell_types <- as.character(cell_types)
cell_types <- gsub("[0-9]+$", "", cell_types)
cell_types <- trimws(cell_types)

sc_matrix <- GetAssayData(cd45_cells, layer = "counts")

if (length(cell_types) != ncol(sc_matrix)) {
  if (length(cell_types) == nrow(cd45_cells@meta.data)) {
    cell_types <- cell_types[colnames(sc_matrix)]
  } else {
    stop("Cannot reconcile the mismatch between cell types and matrix dimensions")
  }
}

results_dir <- "/content/drive/MyDrive/BayesPrism/results_annotation"
if (!dir.exists(results_dir)) dir.create(results_dir, recursive = TRUE)

saveRDS(cd45_cells, file = file.path(results_dir, "cd45_seurat_object.rds"))
saveRDS(sc_matrix, file = file.path(results_dir, "cd45_counts.rds"))
saveRDS(cell_types, file = file.path(results_dir, "cd45_cell_types.rds"))

DimPlot(cd45_cells, group.by = paste0("cell_type_", best_ref, "_main"), label = TRUE)

In [ ]:
%%R
# ENHANCED SINGLER ANNOTATION WITH QUALITY CONTROL
# Optimized for comprehensive BayesPrism deconvolution

library(SingleR)
library(celldex)
library(Seurat)
library(dplyr)
library(ggplot2)

# Set up directories for organized output
results_dir <- "/content/drive/MyDrive/BayesPrism/results_annotation"
singler_dir <- file.path(results_dir, "singler_detailed_results")
if (!dir.exists(singler_dir)) dir.create(singler_dir, recursive = TRUE)

cat("=== ENHANCED SINGLER ANNOTATION PIPELINE ===\n")
cat("Results will be saved to:", singler_dir, "\n")

# 1. LOAD REFERENCES WITH ENHANCED ERROR HANDLING

cat("\n=== LOADING REFERENCE DATASETS ===\n")

# Define reference datasets with metadata
reference_info <- list(
  DICE = list(
    name = "Database of Immune Cell Expression",
    func = celldex::DatabaseImmuneCellExpressionData,
    description = "Flow-sorted immune cell populations",
    best_for = "General immune cell types"
  ),
  Monaco = list(
    name = "Monaco Immune Data",
    func = celldex::MonacoImmuneData,
    description = "Mass cytometry immune cell atlas",
    best_for = "Detailed immune subtypes, T/B cells"
  ),
  Novershtern = list(
    name = "Novershtern Hematopoietic Data",
    func = celldex::NovershternHematopoieticData,
    description = "Hematopoietic differentiation hierarchy",
    best_for = "Hematopoietic progenitors, stem cells"
  ),
  Blueprint = list(
    name = "Blueprint Encode Data",
    func = celldex::BlueprintEncodeData,
    description = "Broad immune and non-immune cell types",
    best_for = "Mixed tissue analysis"
  )
)

# Load references with enhanced error handling
references <- list()
reference_metadata <- list()

for (ref_name in names(reference_info)) {
  cat("Loading", ref_name, "reference...\n")

  ref_result <- tryCatch({
    ref_data <- reference_info[[ref_name]]$func()

    # Validate reference
    if (is.null(ref_data) || ncol(ref_data) == 0) {
      stop("Empty or invalid reference data")
    }

    cat("  ✓ Loaded:", ref_name, "\n")
    cat("    Genes:", nrow(ref_data), "\n")
    cat("    Samples:", ncol(ref_data), "\n")
    cat("    Main labels:", length(unique(ref_data$label.main)), "\n")
    cat("    Fine labels:", length(unique(ref_data$label.fine)), "\n")

    # Store metadata
    reference_metadata[[ref_name]] <- list(
      n_genes = nrow(ref_data),
      n_samples = ncol(ref_data),
      n_main_labels = length(unique(ref_data$label.main)),
      n_fine_labels = length(unique(ref_data$label.fine)),
      main_labels = unique(ref_data$label.main),
      fine_labels = unique(ref_data$label.fine)
    )

    ref_data

  }, error = function(e) {
    cat("  ✗ Failed to load", ref_name, ":", e$message, "\n")
    NULL
  })

  if (!is.null(ref_result)) {
    references[[ref_name]] <- ref_result
  }
}

cat("\nSuccessfully loaded", length(references), "out of", length(reference_info), "references\n")

# Save reference metadata
saveRDS(reference_metadata, file.path(singler_dir, "reference_metadata.rds"))

# 2. PREPARE QUERY DATA WITH OPTIMIZATION

cat("\n=== PREPARING QUERY DATA ===\n")

# Get expression data (assuming cd45_cells is already loaded)
cd45_expr <- GetAssayData(cd45_cells, layer = "data")

cat("Query data dimensions:\n")
cat("  Genes:", nrow(cd45_expr), "\n")
cat("  Cells:", ncol(cd45_expr), "\n")

if (ncol(cd45_expr) < 100) {
  warning("Low number of cells (", ncol(cd45_expr), "). Results may be unreliable.")
}

if (nrow(cd45_expr) < 1000) {
  warning("Low number of genes (", nrow(cd45_expr), "). Consider using more genes.")
}

# 3. ENHANCED SINGLER ANNOTATION WITH CUSTOM PARAMETERS

cat("\n=== RUNNING ENHANCED SINGLER ANNOTATION ===\n")

# Custom SingleR parameters optimized for immune cells
singler_params <- list(
  # For main labels - more permissive
  main = list(
    de.method = "wilcox",      # Robust for immune cells
    de.n = 50,                 # Top 50 genes per label
    fine.tune = TRUE,          # Enable fine-tuning
    tune.thresh = 0.05,        # Threshold for fine-tuning
    sd.thresh = 1,             # Standard deviation threshold
    prune = TRUE               # Remove low-quality assignments
  ),

  # For fine labels - more stringent
  fine = list(
    de.method = "wilcox",
    de.n = 20,                 # Fewer genes for fine labels
    fine.tune = TRUE,
    tune.thresh = 0.1,         # More stringent threshold
    sd.thresh = 1.5,           # Higher SD threshold
    prune = TRUE
  )
)

# Initialize results storage
singler_results <- list()
annotation_summary <- data.frame()

# Process each reference
for (ref_name in names(references)) {
  cat("\nProcessing reference:", ref_name, "\n")
  ref <- references[[ref_name]]

  # Calculate gene overlap
  common_genes <- intersect(rownames(cd45_expr), rownames(ref))
  overlap_pct <- length(common_genes) / nrow(ref) * 100

  cat("  Gene overlap:", length(common_genes), "genes (",
      round(overlap_pct, 1), "% of reference)\n")

  if (length(common_genes) < 100) {
    cat("  ⚠ Warning: Low gene overlap. Skipping", ref_name, "\n")
    next
  }

  # MAIN LEVEL ANNOTATION
  cat("  Running main-level annotation...\n")
  singler_main <- tryCatch({
    SingleR(
      test = cd45_expr,
      ref = ref,
      labels = ref$label.main,
      method = singler_params$main$de.method,
      de.n = singler_params$main$de.n,
      fine.tune = singler_params$main$fine.tune,
      tune.thresh = singler_params$main$tune.thresh,
      sd.thresh = singler_params$main$sd.thresh,
      prune = singler_params$main$prune
    )
  }, error = function(e) {
    cat("    ✗ Main annotation failed:", e$message, "\n")
    NULL
  })

  # FINE LEVEL ANNOTATION
  cat("  Running fine-level annotation...\n")
  singler_fine <- tryCatch({
    SingleR(
      test = cd45_expr,
      ref = ref,
      labels = ref$label.fine,
      method = singler_params$fine$de.method,
      de.n = singler_params$fine$de.n,
      fine.tune = singler_params$fine$fine.tune,
      tune.thresh = singler_params$fine$tune.thresh,
      sd.thresh = singler_params$fine$sd.thresh,
      prune = singler_params$fine$prune
    )
  }, error = function(e) {
    cat("    ✗ Fine annotation failed:", e$message, "\n")
    NULL
  })

  # Store results if successful
  if (!is.null(singler_main) && !is.null(singler_fine)) {

    # Add to Seurat object
    cd45_cells[[paste0("cell_type_", tolower(ref_name), "_main")]] <- singler_main$labels
    cd45_cells[[paste0("cell_type_", tolower(ref_name), "_fine")]] <- singler_fine$labels

    # Add confidence scores
    cd45_cells[[paste0("confidence_", tolower(ref_name), "_main")]] <- singler_main$scores[cbind(1:nrow(singler_main$scores), max.col(singler_main$scores))]
    cd45_cells[[paste0("confidence_", tolower(ref_name), "_fine")]] <- singler_fine$scores[cbind(1:nrow(singler_fine$scores), max.col(singler_fine$scores))]

    # Store detailed results
    singler_results[[ref_name]] <- list(
      main = singler_main,
      fine = singler_fine,
      reference_info = reference_info[[ref_name]],
      gene_overlap = length(common_genes),
      overlap_percentage = overlap_pct
    )

    # Calculate quality metrics
    main_confidence <- mean(singler_main$scores[cbind(1:nrow(singler_main$scores), max.col(singler_main$scores))])
    fine_confidence <- mean(singler_fine$scores[cbind(1:nrow(singler_fine$scores), max.col(singler_fine$scores))])

    main_pruned <- sum(is.na(singler_main$pruned.labels))
    fine_pruned <- sum(is.na(singler_fine$pruned.labels))

    # Add to summary
    summary_row <- data.frame(
      reference = ref_name,
      gene_overlap = length(common_genes),
      overlap_pct = overlap_pct,
      main_labels = length(unique(singler_main$labels[!is.na(singler_main$labels)])),
      fine_labels = length(unique(singler_fine$labels[!is.na(singler_fine$labels)])),
      main_confidence = main_confidence,
      fine_confidence = fine_confidence,
      main_pruned = main_pruned,
      fine_pruned = fine_pruned,
      main_pruned_pct = main_pruned / ncol(cd45_expr) * 100,
      fine_pruned_pct = fine_pruned / ncol(cd45_expr) * 100,
      stringsAsFactors = FALSE
    )

    annotation_summary <- rbind(annotation_summary, summary_row)

    cat("    ✓ Completed", ref_name, "\n")
    cat("      Main confidence:", round(main_confidence, 3), "\n")
    cat("      Fine confidence:", round(fine_confidence, 3), "\n")
    cat("      Pruned cells: Main =", main_pruned, ", Fine =", fine_pruned, "\n")

    # Save individual results
    saveRDS(singler_main, file.path(singler_dir, paste0("singler_", tolower(ref_name), "_main_detailed.rds")))
    saveRDS(singler_fine, file.path(singler_dir, paste0("singler_", tolower(ref_name), "_fine_detailed.rds")))
  }
}

# 4. QUALITY CONTROL AND CONSENSUS ANALYSIS

cat("\n=== QUALITY CONTROL ANALYSIS ===\n")

# Create consensus annotation (best reference per cell)
if (nrow(annotation_summary) > 1) {

  # Find best reference based on overall confidence
  best_main_ref <- annotation_summary$reference[which.max(annotation_summary$main_confidence)]
  best_fine_ref <- annotation_summary$reference[which.max(annotation_summary$fine_confidence)]

  cat("Best reference for main annotation:", best_main_ref, "\n")
  cat("Best reference for fine annotation:", best_fine_ref, "\n")

  # Create consensus annotation
  cd45_cells$consensus_main <- cd45_cells[[paste0("cell_type_", tolower(best_main_ref), "_main")]]
  cd45_cells$consensus_fine <- cd45_cells[[paste0("cell_type_", tolower(best_fine_ref), "_fine")]]

  # Calculate annotation agreement between references
  if (length(singler_results) >= 2) {
    ref_names <- names(singler_results)[1:2]
    main_agreement <- mean(cd45_cells[[paste0("cell_type_", tolower(ref_names[1]), "_main")]] ==
                          cd45_cells[[paste0("cell_type_", tolower(ref_names[2]), "_main")]], na.rm = TRUE)

    cat("Agreement between top 2 references (main):", round(main_agreement * 100, 1), "%\n")
  }
}

# 5. VISUALIZATION AND REPORTING

cat("\n=== CREATING VISUALIZATION PLOTS ===\n")

# Create UMAP plots for each reference
plot_list <- list()

for (ref_name in names(singler_results)) {
  # Main level plot
  p_main <- DimPlot(cd45_cells,
                   group.by = paste0("cell_type_", tolower(ref_name), "_main"),
                   label = TRUE, label.size = 3, pt.size = 0.5) +
    ggtitle(paste("SingleR Annotation -", ref_name, "(Main)")) +
    theme(legend.position = "none")

  # Fine level plot
  p_fine <- DimPlot(cd45_cells,
                   group.by = paste0("cell_type_", tolower(ref_name), "_fine"),
                   label = FALSE, pt.size = 0.5) +
    ggtitle(paste("SingleR Annotation -", ref_name, "(Fine)")) +
    theme(legend.position = "none")

  # Save plots
  ggsave(file.path(singler_dir, paste0("UMAP_", tolower(ref_name), "_main.png")),
         p_main, width = 10, height = 8, dpi = 300)
  ggsave(file.path(singler_dir, paste0("UMAP_", tolower(ref_name), "_fine.png")),
         p_fine, width = 10, height = 8, dpi = 300)

  plot_list[[paste0(ref_name, "_main")]] <- p_main
  plot_list[[paste0(ref_name, "_fine")]] <- p_fine
}

# Create confidence score plots
if (length(singler_results) > 0) {
  for (ref_name in names(singler_results)) {
    confidence_col <- paste0("confidence_", tolower(ref_name), "_main")
    if (confidence_col %in% colnames(cd45_cells@meta.data)) {

      p_conf <- FeaturePlot(cd45_cells, features = confidence_col, pt.size = 0.5) +
        scale_color_viridis_c(name = "Confidence") +
        ggtitle(paste("Annotation Confidence -", ref_name))

      ggsave(file.path(singler_dir, paste0("confidence_", tolower(ref_name), ".png")),
             p_conf, width = 8, height = 6, dpi = 300)
    }
  }
}

# 6. SAVE COMPREHENSIVE RESULTS

cat("\n=== SAVING COMPREHENSIVE RESULTS ===\n")

# Save updated Seurat object
saveRDS(cd45_cells, file.path(singler_dir, "cd45_cells_with_singler_annotations.rds"))

# Save all SingleR results
saveRDS(singler_results, file.path(singler_dir, "all_singler_results.rds"))

# Save annotation summary
write.csv(annotation_summary, file.path(singler_dir, "annotation_quality_summary.csv"), row.names = FALSE)

# Create detailed report
report_data <- list(
  timestamp = Sys.time(),
  query_info = list(
    n_cells = ncol(cd45_cells),
    n_genes = nrow(cd45_expr),
    seurat_clusters = length(unique(cd45_cells$seurat_clusters))
  ),
  references_used = names(singler_results),
  annotation_summary = annotation_summary,
  singler_parameters = singler_params,
  files_created = list(
    main_results = paste0("singler_", tolower(names(singler_results)), "_main_detailed.rds"),
    fine_results = paste0("singler_", tolower(names(singler_results)), "_fine_detailed.rds"),
    plots_main = paste0("UMAP_", tolower(names(singler_results)), "_main.png"),
    plots_fine = paste0("UMAP_", tolower(names(singler_results)), "_fine.png"),
    confidence_plots = paste0("confidence_", tolower(names(singler_results)), ".png")
  )
)

saveRDS(report_data, file.path(singler_dir, "singler_analysis_report.rds"))

# 7. INTEGRATION WITH BAYESPRISM PIPELINE

cat("\n=== PREPARING FOR BAYESPRISM INTEGRATION ===\n")

# Extract the best annotations for BayesPrism
if (exists("best_main_ref")) {
  col_main <- paste0("cell_type_", tolower(best_main_ref), "_main")
  col_conf <- paste0("confidence_", tolower(best_main_ref), "_main")


  best_cell_types   <- cd45_cells@meta.data[[col_main]]
  best_confidences  <- cd45_cells@meta.data[[col_conf]]

  n_cells <- length(best_cell_types)
  confidence_threshold <- 0.5
  high_conf_mask <- (best_confidences > confidence_threshold) & !is.na(best_cell_types)

  cat("Cells passing confidence threshold (", confidence_threshold, "): ",
      sum(high_conf_mask), " / ", n_cells, "\n", sep = "")

  final_cell_types <- best_cell_types
  final_cell_types[!high_conf_mask] <- "Unassigned"

  saveRDS(final_cell_types, file.path(singler_dir, "bayesprism_cell_types.rds"))
  saveRDS(best_confidences, file.path(singler_dir, "bayesprism_confidences.rds"))

  bayesprism_summary <- sort(table(final_cell_types), decreasing = TRUE)
  write.csv(
    data.frame(cell_type = names(bayesprism_summary),
               count = as.integer(bayesprism_summary)),
    file.path(singler_dir, "bayesprism_cell_type_counts.csv"),
    row.names = FALSE
  )

  cat("Cell type distribution for BayesPrism:\n")
  print(bayesprism_summary)
}


# 8. FINAL SUMMARY

cat("\n ENHANCED SINGLER ANNOTATION COMPLETED! \n")

cat("Results saved to:", singler_dir, "\n")
cat("References processed:", length(singler_results), "\n")
cat("Total annotations added to Seurat object:", sum(grepl("cell_type_", colnames(cd45_cells@meta.data))), "\n")

if (exists("best_main_ref")) {
  cat("Recommended reference for downstream analysis:", best_main_ref, "\n")
  cat("Cell types ready for BayesPrism: bayesprism_cell_types.rds\n")
}

cat("\nFiles created:\n")
cat("- Detailed SingleR results (", length(singler_results) * 2, " files)\n", sep="")
cat("- UMAP visualization plots (", length(singler_results) * 2, " files)\n", sep="")
cat("- Confidence score plots (", length(singler_results), " files)\n", sep="")
cat("- Quality summary (annotation_quality_summary.csv)\n")
cat("- Updated Seurat object (cd45_cells_with_singler_annotations.rds)\n")
cat("- Analysis report (singler_analysis_report.rds)\n")
cat("- BayesPrism integration files (3 files)\n")

cat("\n Ready for your sophisticated BayesPrism deconvolution pipeline! \n")

In [ ]:
%%R
# Use Monaco main annotation if available (changed to _main)
best_ref <- ifelse("monaco" %in% tolower(names(references)), "monaco", "dice")

# Debug: Check the structure of cd45_cells
cat("Number of cells in cd45_cells:", ncol(cd45_cells), "\n")
cat("Available metadata columns:", names(cd45_cells@meta.data), "\n")

# Get the cell type labels
cell_type_col <- paste0("cell_type_", best_ref, "_main")
if (!cell_type_col %in% names(cd45_cells@meta.data)) {
  stop(paste("Column", cell_type_col, "not found in cd45_cells metadata"))
}

# Extract cell types
cell_types <- cd45_cells@meta.data[[cell_type_col]]
cat("Initial length of cell_types:", length(cell_types), "\n")

# Ensure cell types are character vector
cell_types <- as.character(cell_types)
cell_types <- gsub("[0-9]+$", "", cell_types)
cell_types <- trimws(cell_types)

# Extract count matrix for BayesPrism
sc_matrix <- GetAssayData(cd45_cells, slot = "counts")

# Debug: Check dimensions
cat("Number of cells in sc_matrix:", ncol(sc_matrix), "\n")
cat("Length of cell_types after processing:", length(cell_types), "\n")

# Make sure they match exactly
if (length(cell_types) != ncol(sc_matrix)) {
  cat("Mismatch detected!\n")
  cat("cell_types length:", length(cell_types), "\n")
  cat("sc_matrix columns:", ncol(sc_matrix), "\n")

  if (length(cell_types) == nrow(cd45_cells@meta.data)) {
    cat("Cell types length matches metadata rows, extracting in proper order...\n")
    # Ensure cell types are in the same order as the cells in the count matrix
    cell_types <- cell_types[colnames(sc_matrix)]
    cat("Final cell_types length:", length(cell_types), "\n")
  } else {
    stop("Cannot reconcile the mismatch between cell types and matrix dimensions")
  }
}

results_dir <- "/content/drive/MyDrive/BayesPrism/results_annotation"
if (!dir.exists(results_dir)) dir.create(results_dir, recursive = TRUE)

# Save files
saveRDS(cd45_cells, file = file.path(results_dir, "cd45_seurat_object.rds"))
saveRDS(sc_matrix, file = file.path(results_dir, "cd45_counts.rds"))
saveRDS(cell_types, file = file.path(results_dir, "cd45_cell_types.rds"))

# Summary
cat("BayesPrism CD45+ Reference Prepared\n")
cat("Number of cells:", ncol(sc_matrix), "\n")
cat("Number of genes:", nrow(sc_matrix), "\n")
print(table(cell_types))

# Visualization
DimPlot(cd45_cells, group.by = paste0("cell_type_", best_ref, "_main"), label = TRUE)

In [ ]:
%%R
# Load ggplot2 for saving TIFFs
library(ggplot2)

# Define output directory
results_dir <- "/content/drive/MyDrive/BayesPrism/results_annotation"
if (!dir.exists(results_dir)) dir.create(results_dir, recursive = TRUE)

# Loop over all references and save both main and fine UMAPs as TIFF
for (ref_name in names(references)) {
  ref_lower <- tolower(ref_name)
  main_col <- paste0("cell_type_", ref_lower, "_main")
  fine_col <- paste0("cell_type_", ref_lower, "_fine")

  # Plot and save main-level annotation
  if (main_col %in% colnames(cd45_cells@meta.data)) {
    p_main <- DimPlot(cd45_cells, group.by = main_col, label = TRUE) +
      ggtitle(paste("UMAP -", ref_name, "(main labels)"))

    tiff_filename_main <- file.path(results_dir, paste0("UMAP_", ref_lower, "_main.tiff"))
    tiff(tiff_filename_main, width = 8, height = 6, units = "in", res = 300)
    print(p_main)
    dev.off()
  }

  # Plot and save fine-level annotation
  if (fine_col %in% colnames(cd45_cells@meta.data)) {
    p_fine <- DimPlot(cd45_cells, group.by = fine_col, label = FALSE) +
      ggtitle(paste("UMAP -", ref_name, "(fine labels)"))

    tiff_filename_fine <- file.path(results_dir, paste0("UMAP_", ref_lower, "_fine.tiff"))
    tiff(tiff_filename_fine, width = 8, height = 6, units = "in", res = 300)
    print(p_fine)
    dev.off()
  }
}

# Define output directory
results_dir <- "/content/drive/MyDrive/BayesPrism/results_annotation"
if (!dir.exists(results_dir)) dir.create(results_dir, recursive = TRUE)

# Initialize lists to store results
main_tables <- list()
fine_tables <- list()

# Loop through each reference to extract counts
for (ref_name in names(references)) {
  ref_lower <- tolower(ref_name)

  main_col <- paste0("cell_type_", ref_lower, "_main")
  fine_col <- paste0("cell_type_", ref_lower, "_fine")

  # Main-level annotation counts
  if (main_col %in% colnames(cd45_cells@meta.data)) {
    tbl_main <- as.data.frame(table(cd45_cells[[main_col]]))
    colnames(tbl_main) <- c("Cell_Type", "Count")
    main_tables[[ref_name]] <- tbl_main
    write.csv(tbl_main, file = file.path(results_dir, paste0("annotation_summary_", ref_lower, "_main.csv")), row.names = FALSE)
  }

  # Fine-level annotation counts
  if (fine_col %in% colnames(cd45_cells@meta.data)) {
    tbl_fine <- as.data.frame(table(cd45_cells[[fine_col]]))
    colnames(tbl_fine) <- c("Cell_Type", "Count")
    fine_tables[[ref_name]] <- tbl_fine
    write.csv(tbl_fine, file = file.path(results_dir, paste0("annotation_summary_", ref_lower, "_fine.csv")), row.names = FALSE)
  }
}

cat("Annotation summary tables saved to:", results_dir, "\n")


In [ ]:
%%R

setwd("/content/drive/MyDrive/BayesPrism")

# Load required libraries
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
if (!requireNamespace("BayesPrism", quietly = TRUE)) {
  if (!requireNamespace("devtools", quietly = TRUE)) install.packages("devtools")
  devtools::install_github("Danko-Lab/BayesPrism/BayesPrism")
}
if (!requireNamespace("Seurat", quietly = TRUE)) install.packages("Seurat")
if (!requireNamespace("biomaRt", quietly = TRUE)) BiocManager::install("biomaRt")
if (!requireNamespace("dplyr", quietly = TRUE)) install.packages("dplyr")
if (!requireNamespace("ggplot2", quietly = TRUE)) install.packages("ggplot2")
if (!requireNamespace("pheatmap", quietly = TRUE)) install.packages("pheatmap")

library(BayesPrism)
library(Seurat)
library(biomaRt)
library(dplyr)
library(ggplot2)
library(pheatmap)

In [ ]:
%%R
# File paths
bulk_normal_path <- "/content/drive/MyDrive/BayesPrism/data/pdac_normal_weight.csv"
bulk_overweight_path <- "/content/drive/MyDrive/BayesPrism/data/pdac_over_weight.csv"
bulk_obese_path <- "/content/drive/MyDrive/BayesPrism/data/pdac_Obese.csv"
sc_matrix_path <- "/content/drive/MyDrive/BayesPrism/results_annotation/cd45_counts.rds"
cell_types_path <- "/content/drive/MyDrive/BayesPrism/results_annotation/cd45_cell_types.rds"

In [ ]:
%%R
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

BiocManager::install(c("org.Hs.eg.db", "AnnotationDbi"))

library(org.Hs.eg.db)
library(AnnotationDbi)

In [ ]:
%%R
# STEP 1: Load and preprocess bulk RNA-seq data

# Load bulk RNA-seq data
read_bulk_data <- function(file_path) {
  bulk_data <- read.csv(file_path, header = TRUE, check.names = FALSE)
  rownames(bulk_data) <- bulk_data[, 1]
  bulk_data <- bulk_data[, -1]
  bulk_data <- as.matrix(bulk_data)
  mode(bulk_data) <- "numeric"
  return(bulk_data)
}

bulk_normal <- read_bulk_data(bulk_normal_path)
bulk_overweight <- read_bulk_data(bulk_overweight_path)
bulk_obese <- read_bulk_data(bulk_obese_path)

# Clean Ensembl IDs (remove version numbers)
extract_ensembl_ids <- function(ids) gsub("\\.\\d+$", "", ids)
rownames(bulk_normal) <- extract_ensembl_ids(rownames(bulk_normal))
rownames(bulk_overweight) <- extract_ensembl_ids(rownames(bulk_overweight))
rownames(bulk_obese) <- extract_ensembl_ids(rownames(bulk_obese))

# Merge all samples
bulk_combined <- cbind(bulk_normal, bulk_overweight, bulk_obese)
sample_groups <- c(rep("normal", ncol(bulk_normal)),
                   rep("overweight", ncol(bulk_overweight)),
                   rep("obese", ncol(bulk_obese)))
names(sample_groups) <- colnames(bulk_combined)
bulk_combined <- bulk_combined[rowSums(bulk_combined) > 0, ]

# Load CD45+ single-cell reference
sc_matrix <- readRDS(sc_matrix_path)
cell_types <- readRDS(cell_types_path)

# Harmonize gene symbols using org.Hs.eg.db (offline annotation)
map_ensembl_to_symbol <- function(ensembl_ids) {
  cat("Mapping", length(ensembl_ids), "Ensembl IDs to gene symbols using org.Hs.eg.db...\n")

  # Remove version numbers if still present
  clean_ids <- gsub("\\.\\d+$", "", ensembl_ids)

  # Map Ensembl IDs to gene symbols
  gene_symbols <- mapIds(org.Hs.eg.db,
                        keys = clean_ids,
                        column = "SYMBOL",
                        keytype = "ENSEMBL",
                        multiVals = "first")  # Take first symbol if multiple exist

  # Report mapping statistics
  mapped_count <- sum(!is.na(gene_symbols))
  cat("Successfully mapped", mapped_count, "out of", length(ensembl_ids),
      "genes (", round(mapped_count/length(ensembl_ids)*100, 1), "%)\n")

  # Return named vector
  names(gene_symbols) <- clean_ids
  return(gene_symbols)
}

# Perform the mapping
ensembl_to_symbol <- map_ensembl_to_symbol(rownames(bulk_combined))

# Create new matrix with gene symbols as rownames
bulk_combined_with_symbols <- bulk_combined
rownames(bulk_combined_with_symbols) <- ensembl_to_symbol[rownames(bulk_combined)]

# Remove genes that couldn't be mapped (NA or empty symbols)
genes_to_keep <- !is.na(rownames(bulk_combined_with_symbols)) &
                 rownames(bulk_combined_with_symbols) != ""
bulk_combined_with_symbols <- bulk_combined_with_symbols[genes_to_keep, ]

cat("After removing unmapped genes:", nrow(bulk_combined_with_symbols), "genes remaining\n")

# Remove duplicated symbols by retaining the one with highest mean expression
if (any(duplicated(rownames(bulk_combined_with_symbols)))) {
  cat("Handling duplicated gene symbols...\n")

  dups <- unique(rownames(bulk_combined_with_symbols)[duplicated(rownames(bulk_combined_with_symbols))])
  cat("Found", length(dups), "duplicated gene symbols\n")

  for (gene in dups) {
    idxs <- which(rownames(bulk_combined_with_symbols) == gene)
    mean_expr <- rowMeans(bulk_combined_with_symbols[idxs, , drop = FALSE])
    keep_idx <- idxs[which.max(mean_expr)]
    remove_idxs <- setdiff(idxs, keep_idx)
    bulk_combined_with_symbols <- bulk_combined_with_symbols[-remove_idxs, ]
  }

  cat("After removing duplicates:", nrow(bulk_combined_with_symbols), "genes remaining\n")
}

# Final summary
cat("\nFinal dataset summary:\n")
cat("Dimensions:", nrow(bulk_combined_with_symbols), "genes x", ncol(bulk_combined_with_symbols), "samples\n")
cat("Sample groups:", table(sample_groups), "\n")

# Optional: Save the mapping for future reference
mapping_df <- data.frame(
  ensembl_id = names(ensembl_to_symbol),
  gene_symbol = ensembl_to_symbol,
  stringsAsFactors = FALSE
)
mapping_df <- mapping_df[!is.na(mapping_df$gene_symbol), ]

# write.csv(mapping_df, "ensembl_to_symbol_mapping.csv", row.names = FALSE)

In [ ]:
%%R
# Match genes between bulk and scRNA-seq
common_genes <- intersect(rownames(bulk_combined_with_symbols), rownames(sc_matrix))
top_genes <- names(sort(rowMeans(sc_matrix[common_genes, ]), decreasing = TRUE))[1:18000]
common_genes <- intersect(common_genes, top_genes)

bulk_filtered <- bulk_combined_with_symbols[common_genes, ]
sc_filtered <- sc_matrix[common_genes, ]

# Prepare BayesPrism input - correct transposition
sc_ref <- t(sc_filtered)  # cells x genes
bulk_mix <- t(bulk_filtered)  # samples x genes
cell_type_labels <- as.character(cell_types)

# Add debugging checks
cat("Dimensions of sc_ref (cells x genes):", dim(sc_ref), "\n")
cat("Dimensions of bulk_mix (samples x genes):", dim(bulk_mix), "\n")
cat("Length of cell_type_labels:", length(cell_type_labels), "\n")
cat("Number of unique cell types:", length(unique(cell_type_labels)), "\n")

# Validate shapes
stopifnot(identical(colnames(sc_ref), colnames(bulk_mix)))
stopifnot(length(cell_type_labels) == nrow(sc_ref))

# Deconvolution using BayesPrism
bp <- new.prism(
  reference = sc_ref,
  mixture = bulk_mix,
  input.type = "count",
  cell.type.labels = cell_type_labels,
  cell.state.labels = cell_type_labels,
  key = NULL,
  outlier.cut = 0.01,
  outlier.fraction = 0.05
)

In [ ]:
%%R
# STEP 4: Run BayesPrism


bp <- run.prism(bp,
                n.cores = 8,
                update.gibbs = TRUE, # Run final Gibbs sampling to update theta
                gibbs.control = list(
                  chain.length = 2000,  # Increased from default for publication-quality results
                  burn.in = 500,        # Default burn-in period
                  thinning = 2          # Default thinning to reduce autocorrelation
                ))



In [ ]:
%%R
# Save the results
cat("Saving BayesPrism results...\n")
saveRDS(bp, file = "/content/drive/MyDrive/BayesPrism/results_annotation/bayesprism_results.rds")
# Check the structure of the BayesPrism object
str(bp)
slotNames(bp)

In [ ]:
%%R
# COMPREHENSIVE IMMUNE CELL SUBTYPE REFINEMENT
# T cells + Monocytes + NK cells + Dendritic cells

# Load processed CD45+ cells
results_dir <- "/content/drive/MyDrive/BayesPrism/results_annotation"
cd45_cells <- readRDS(file.path(results_dir, "cd45_seurat_object.rds"))
sc_matrix <- readRDS(file.path(results_dir, "cd45_counts.rds"))
cell_types <- readRDS(file.path(results_dir, "cd45_cell_types.rds"))

cat("Starting comprehensive immune cell refinement...\n")
cat("Current cell type distribution:\n")
print(table(cell_types))


cat("\nAvailable fine-grained annotations:\n")
fine_cols <- grep("_fine$", colnames(cd45_cells@meta.data), value = TRUE)
for(col in fine_cols) {
  cat("\n", col, ":\n")
  print(head(table(cd45_cells@meta.data[[col]]), 10))
}

# STRATEGY 1: Use existing Monaco fine annotations

refine_immune_cells_with_monaco <- function(cd45_cells, current_cell_types) {

  # Get Monaco fine annotations
  monaco_fine <- cd45_cells$cell_type_monaco_fine

  # Create refined cell types starting with current annotations
  refined_types <- current_cell_types

  # Define comprehensive mapping from Monaco fine to  desired categories
  monaco_mapping <- list(
    # T CELL SUBTYPES
    "T cells CD8+" = "CD8+ T cells",
    "T cells CD8+ naive" = "CD8+ T naive",
    "T cells CD8+ effector memory" = "CD8+ T effector memory",
    "T cells CD8+ central memory" = "CD8+ T central memory",
    "T cells CD8+ effector memory RA+" = "CD8+ T effector",
    "T cells CD8+ terminal effector" = "CD8+ T effector",
    "T cells CD4+" = "CD4+ T cells",
    "T cells CD4+ naive" = "CD4+ T naive",
    "T cells CD4+ memory" = "CD4+ T memory",
    "T cells CD4+ effector memory" = "CD4+ T effector memory",
    "T cells CD4+ central memory" = "CD4+ T central memory",
    "T cells CD4+ Th1" = "CD4+ T Th1",
    "T cells CD4+ Th2" = "CD4+ T Th2",
    "T cells CD4+ Th17" = "CD4+ T Th17",
    "T cells CD4+ follicular helper" = "CD4+ T Tfh",
    "T regulatory cells" = "CD4+ T reg",
    "Tregs" = "CD4+ T reg",
    "T cells gamma delta" = "T cells gamma delta",
    "MAIT cells" = "MAIT cells",
    "NKT cells" = "NKT cells",

    # MONOCYTE SUBTYPES
    "Monocytes" = "Monocytes",
    "Monocytes CD14+" = "Monocytes classical",
    "Monocytes CD16+" = "Monocytes non-classical",
    "Monocytes CD14+ CD16+" = "Monocytes intermediate",
    "Monocytes classical" = "Monocytes classical",
    "Monocytes non-classical" = "Monocytes non-classical",
    "Monocytes intermediate" = "Monocytes intermediate",

    # NK CELL SUBTYPES
    "NK cells" = "NK cells",
    "NK cells CD56bright" = "NK cells CD56bright",
    "NK cells CD56dim" = "NK cells CD56dim",
    "NK cells adaptive" = "NK cells adaptive",
    "NK cells cytotoxic" = "NK cells cytotoxic",

    # DENDRITIC CELL SUBTYPES
    "Dendritic cells" = "Dendritic cells",
    "mDCs" = "mDC",
    "pDCs" = "pDC",
    "cDCs" = "cDC",
    "DC1" = "cDC1",
    "DC2" = "cDC2",
    "Dendritic cells myeloid" = "mDC",
    "Dendritic cells plasmacytoid" = "pDC",
    "Conventional dendritic cells" = "cDC",

    # B CELL SUBTYPES (bonus)
    "B cells" = "B cells",
    "B cells naive" = "B cells naive",
    "B cells memory" = "B cells memory",
    "Plasma cells" = "Plasma cells",

    # OTHER
    "Basophils" = "Basophils",
    "Eosinophils" = "Eosinophils",
    "Neutrophils" = "Neutrophils"
  )

  # Apply Monaco-based refinement
  for(i in seq_along(refined_types)) {
    monaco_annotation <- monaco_fine[i]


    if(!is.na(monaco_annotation) && monaco_annotation %in% names(monaco_mapping)) {
      refined_types[i] <- monaco_mapping[[monaco_annotation]]
    }

    else if(!is.na(monaco_annotation)) {
      # T cells
      if(grepl("T cells", refined_types[i])) {
        if(grepl("CD8", monaco_annotation, ignore.case = TRUE)) {
          if(grepl("naive", monaco_annotation, ignore.case = TRUE)) {
            refined_types[i] <- "CD8+ T naive"
          } else if(grepl("effector", monaco_annotation, ignore.case = TRUE)) {
            refined_types[i] <- "CD8+ T effector"
          } else if(grepl("memory", monaco_annotation, ignore.case = TRUE)) {
            refined_types[i] <- "CD8+ T memory"
          } else {
            refined_types[i] <- "CD8+ T cells"
          }
        }
        else if(grepl("CD4", monaco_annotation, ignore.case = TRUE)) {
          if(grepl("naive", monaco_annotation, ignore.case = TRUE)) {
            refined_types[i] <- "CD4+ T naive"
          } else if(grepl("Th1", monaco_annotation, ignore.case = TRUE)) {
            refined_types[i] <- "CD4+ T Th1"
          } else if(grepl("Th2", monaco_annotation, ignore.case = TRUE)) {
            refined_types[i] <- "CD4+ T Th2"
          } else if(grepl("Th17", monaco_annotation, ignore.case = TRUE)) {
            refined_types[i] <- "CD4+ T Th17"
          } else if(grepl("reg", monaco_annotation, ignore.case = TRUE)) {
            refined_types[i] <- "CD4+ T reg"
          } else {
            refined_types[i] <- "CD4+ T cells"
          }
        }
      }
      # Monocytes
      else if(grepl("Monocytes", refined_types[i])) {
        if(grepl("CD14.*CD16", monaco_annotation, ignore.case = TRUE)) {
          refined_types[i] <- "Monocytes intermediate"
        } else if(grepl("CD14", monaco_annotation, ignore.case = TRUE)) {
          refined_types[i] <- "Monocytes classical"
        } else if(grepl("CD16", monaco_annotation, ignore.case = TRUE)) {
          refined_types[i] <- "Monocytes non-classical"
        }
      }
      # NK cells
      else if(grepl("NK", refined_types[i])) {
        if(grepl("CD56bright", monaco_annotation, ignore.case = TRUE)) {
          refined_types[i] <- "NK cells CD56bright"
        } else if(grepl("CD56dim", monaco_annotation, ignore.case = TRUE)) {
          refined_types[i] <- "NK cells CD56dim"
        }
      }
      # Dendritic cells
      else if(grepl("Dendritic", refined_types[i])) {
        if(grepl("myeloid|mDC", monaco_annotation, ignore.case = TRUE)) {
          refined_types[i] <- "mDC"
        } else if(grepl("plasmacytoid|pDC", monaco_annotation, ignore.case = TRUE)) {
          refined_types[i] <- "pDC"
        }
      }
    }
  }

  return(refined_types)
}



In [ ]:
%%R

# Apply the Monaco refinement function
cat("Applying Monaco refinement function...\n")
refined_cell_types_step1 <- refine_immune_cells_with_monaco(cd45_cells, cell_types)

cat("After Monaco-based refinement:\n")
print(table(refined_cell_types_step1))

# Save Monaco results
output_dir <- "/content/drive/MyDrive/BayesPrism/processed_data"
if (!dir.exists(output_dir)) dir.create(output_dir, recursive = TRUE)

# Save Monaco-refined results
saveRDS(refined_cell_types_step1, file.path(output_dir, "monaco_refined_cell_types.rds"))
saveRDS(sc_matrix, file.path(output_dir, "sc_matrix_for_refinement.rds"))

# Create Monaco comparison table
monaco_comparison_df <- data.frame(
  cell_barcode = colnames(cd45_cells),
  original_main = cell_types,
  monaco_fine = cd45_cells$cell_type_monaco_fine,
  monaco_refined = refined_cell_types_step1,
  stringsAsFactors = FALSE
)

write.csv(monaco_comparison_df, file.path(output_dir, "monaco_refinement_comparison.csv"), row.names = FALSE)

cat("\nMonaco refinement completed and saved!\n")
cat("Files saved:\n")
cat("- Monaco refined types:", file.path(output_dir, "monaco_refined_cell_types.rds"), "\n")
cat("- SC matrix:", file.path(output_dir, "sc_matrix_for_refinement.rds"), "\n")
cat("- Monaco comparison:", file.path(output_dir, "monaco_refinement_comparison.csv"), "\n")

# Show summary of refinement
cat("\nRefinement summary:\n")
cat("Original types:", length(unique(cell_types)), "\n")
cat("Monaco refined types:", length(unique(refined_cell_types_step1)), "\n")

#show remaining
remaining_t_cells <- sum(refined_cell_types_step1 == "T cells")
remaining_monocytes <- sum(refined_cell_types_step1 == "Monocytes")
remaining_nk <- sum(refined_cell_types_step1 == "NK cells")
remaining_dc <- sum(refined_cell_types_step1 == "Dendritic cells")

cat("\nCells still needing marker-based refinement:\n")
cat("- General T cells:", remaining_t_cells, "\n")
cat("- General Monocytes:", remaining_monocytes, "\n")
cat("- General NK cells:", remaining_nk, "\n")
cat("- General Dendritic cells:", remaining_dc, "\n")

if(remaining_t_cells + remaining_monocytes + remaining_nk + remaining_dc > 0) {
  cat("\nYou can now run marker-based refinement for these remaining cells,\n")
  cat("or proceed directly to BayesPrism with current Monaco refinement.\n")
} else {
  cat("\nAll cells have been refined! Ready for BayesPrism.\n")
}

# Visualization
library(ggplot2)
p1 <- DimPlot(cd45_cells, group.by = "cell_type_monaco_main", label = TRUE, pt.size = 0.3) +
      ggtitle("Original Monaco Main") + theme(legend.position = "none")

# Add refined types to Seurat object for visualization
cd45_cells$monaco_refined <- refined_cell_types_step1

p2 <- DimPlot(cd45_cells, group.by = "monaco_refined", label = TRUE, pt.size = 0.3) +
      ggtitle("Monaco Refined") + theme(legend.position = "none")

print(p1)
print(p2)

cat("\nMonaco refinement complete! What would you like to do next?\n")
cat("Option 1: Proceed to BayesPrism with current refined types\n")
cat("Option 2: Run additional marker-based refinement for remaining general cell types\n")

In [ ]:
%%R
# MULTI-REFERENCE REFINEMENT: Monaco + DICE (to capture Tregs)

# Apply Monaco refinement
cat("Applying Monaco refinement...\n")
refined_cell_types_monaco <- refine_immune_cells_with_monaco(cd45_cells, cell_types)

cat("After Monaco refinement:\n")
print(table(refined_cell_types_monaco))

# Add DICE annotations to capture Tregs and other subtypes
cat("\nAdding DICE annotations for Tregs and additional subtypes...\n")
dice_fine <- cd45_cells$cell_type_dice_fine
refined_cell_types_combined <- refined_cell_types_monaco

# Apply DICE-specific refinements
for(i in seq_along(refined_cell_types_combined)) {
  dice_annotation <- dice_fine[i]

  if(!is.na(dice_annotation)) {
    # Capture Tregs from DICE
    if(grepl("TREG", dice_annotation, ignore.case = TRUE)) {
      refined_cell_types_combined[i] <- "CD4+ T reg"
    }
    # Capture additional T helper subtypes
    else if(grepl("Th1", dice_annotation, ignore.case = TRUE)) {
      refined_cell_types_combined[i] <- "CD4+ T Th1"
    }
    else if(grepl("TFH", dice_annotation, ignore.case = TRUE)) {
      refined_cell_types_combined[i] <- "CD4+ T Tfh"
    }
    # Capture monocyte subtypes from DICE
    else if(grepl("CD14\\+", dice_annotation) && !grepl("CD16", dice_annotation)) {
      refined_cell_types_combined[i] <- "Monocytes classical"
    }
    else if(grepl("CD16\\+", dice_annotation)) {
      refined_cell_types_combined[i] <- "Monocytes non-classical"
    }
    # Capture naive B cells
    else if(grepl("B cells, naive", dice_annotation)) {
      refined_cell_types_combined[i] <- "B cells naive"
    }
  }
}

cat("After Monaco + DICE combined refinement:\n")
print(table(refined_cell_types_combined))

# For CD8+ exhausted - marker-based approach
cat("\nAdding marker-based detection for CD8+ exhausted cells...\n")

# Get expression data for exhaustion markers
expr_data <- GetAssayData(cd45_cells, slot = "data")
exhaustion_markers <- c("PDCD1", "HAVCR2", "LAG3", "TIGIT", "TOX")
available_exhaustion_markers <- intersect(exhaustion_markers, rownames(expr_data))

if(length(available_exhaustion_markers) > 0) {
  cat("Found exhaustion markers:", paste(available_exhaustion_markers, collapse = ", "), "\n")

  # Calculate exhaustion score
  if(length(available_exhaustion_markers) == 1) {
    exhaustion_score <- as.numeric(expr_data[available_exhaustion_markers, ])
  } else {
    exhaustion_score <- colMeans(expr_data[available_exhaustion_markers, ], na.rm = TRUE)
  }

  # Apply to CD8+ T cells with high exhaustion score
  cd8_indices <- which(grepl("CD8\\+", refined_cell_types_combined))
  exhaustion_threshold <- quantile(exhaustion_score[cd8_indices], 0.8, na.rm = TRUE)  # Top 20%

  exhausted_indices <- cd8_indices[exhaustion_score[cd8_indices] > exhaustion_threshold &
                                  exhaustion_score[cd8_indices] > 0.5]

  if(length(exhausted_indices) > 0) {
    refined_cell_types_combined[exhausted_indices] <- "CD8+ T exhausted"
    cat("Identified", length(exhausted_indices), "CD8+ T exhausted cells\n")
  }
} else {
  cat("No exhaustion markers found in dataset\n")
}

# Final refined results
refined_cell_types_final <- refined_cell_types_combined

cat("\nFinal multi-reference refined cell type distribution:\n")
print(table(refined_cell_types_final))

# Count specific subtypes of interest
tregs <- sum(refined_cell_types_final == "CD4+ T reg")
cd8_exhausted <- sum(refined_cell_types_final == "CD8+ T exhausted")
mait <- sum(refined_cell_types_final == "MAIT cells")

cat("\nSubtypes of interest for obesity research:\n")
cat("- CD4+ T regulatory cells:", tregs, "\n")
cat("- CD8+ T exhausted cells:", cd8_exhausted, "\n")
cat("- MAIT cells:", mait, "\n")

# Save results
output_dir <- "/content/drive/MyDrive/BayesPrism/processed_data"
if (!dir.exists(output_dir)) dir.create(output_dir, recursive = TRUE)

saveRDS(refined_cell_types_final, file.path(output_dir, "multi_reference_refined_cell_types.rds"))

# Create comparison
comparison_df <- data.frame(
  cell_barcode = colnames(cd45_cells),
  original_main = cell_types,
  monaco_fine = cd45_cells$cell_type_monaco_fine,
  dice_fine = cd45_cells$cell_type_dice_fine,
  monaco_refined = refined_cell_types_monaco,
  final_refined = refined_cell_types_final,
  stringsAsFactors = FALSE
)

write.csv(comparison_df, file.path(output_dir, "multi_reference_refinement_comparison.csv"), row.names = FALSE)

cat("\nMulti-reference refinement completed!\n")
cat("Total refined cell types:", length(unique(refined_cell_types_final)), "\n")

# Show markers used for exhaustion detection
if(length(available_exhaustion_markers) > 0) {
  cat("Exhaustion detection used markers:", paste(available_exhaustion_markers, collapse = ", "), "\n")
}

cat("\nFiles saved:\n")
cat("- Final refined types:", file.path(output_dir, "multi_reference_refined_cell_types.rds"), "\n")
cat("- Comparison table:", file.path(output_dir, "multi_reference_refinement_comparison.csv"), "\n")

In [ ]:
%%R
# Inputs required in env:
#   - cd45_cells (Seurat)
#   - cd45_cells$cell_type_dice_fine (DICE fine column)
suppressPackageStartupMessages({ library(Seurat); library(Matrix) })

if (!exists("cd45_cells")) stop("cd45_cells not found.")
if (!exists("refined_cell_types_monaco")) stop("refined_cell_types_monaco not found.")
if (!"cell_type_dice_fine" %in% colnames(cd45_cells@meta.data)) stop("cd45_cells$cell_type_dice_fine not found.")

names(refined_cell_types_monaco) <- colnames(cd45_cells)

# 1) Guarded DICE integration
dice_fine <- cd45_cells$cell_type_dice_fine
refined_guarded <- refined_cell_types_monaco

for (i in seq_along(refined_guarded)) {
  ann <- dice_fine[i]; if (is.na(ann)) next
  cur <- refined_guarded[i]

  is_T_generic <- grepl("^T cells$", cur)
  is_CD4 <- grepl("^CD4\\+ T", cur)
  is_CD8 <- grepl("^CD8\\+ T", cur)
  is_Mono <- grepl("^Monocytes", cur)
  is_B <- grepl("^B cells", cur)
  is_DC <- grepl("^Dendritic|^mDC$|^pDC$|^cDC", cur)

  # CD4 helper states: only for CD4 T or generic "T cells" with explicit CD4 in DICE
  if (grepl("TREG", ann, TRUE) && (is_CD4 || (is_T_generic && grepl("CD4", ann)))) { refined_guarded[i] <- "CD4+ T reg"; next }
  if (grepl("TFH",  ann, TRUE) && (is_CD4 || (is_T_generic && grepl("CD4", ann)))) { refined_guarded[i] <- "CD4+ T Tfh"; next }
  if (grepl("Th1",  ann, TRUE) && (is_CD4 || (is_T_generic && grepl("CD4", ann)))) { refined_guarded[i] <- "CD4+ T Th1"; next }

  # Monocyte subtypes
  if (is_Mono) {
    if (grepl("CD14\\+\\s*CD16\\+", ann))                    { refined_guarded[i] <- "Monocytes intermediate";    next }
    if (grepl("CD14\\+", ann) && !grepl("CD16", ann))         { refined_guarded[i] <- "Monocytes classical";       next }
    if (grepl("CD16\\+", ann))                                { refined_guarded[i] <- "Monocytes non-classical";   next }
  }

  # B naive
  if (is_B && grepl("B cells,?\\s*naive", ann, TRUE))         { refined_guarded[i] <- "B cells naive"; next }


}

# 2) CD8 exhaustion call (simple marker mean; top 20% + floor)
exhaustion_markers <- c("PDCD1","HAVCR2","LAG3","TIGIT","TOX")
expr_data <- GetAssayData(cd45_cells, slot = "data")

available_exh <- intersect(exhaustion_markers, rownames(expr_data))
if (length(available_exh)) {
  ex_score <- if (length(available_exh) == 1)
    as.numeric(expr_data[available_exh, ])
  else
    colMeans(expr_data[available_exh, ], na.rm = TRUE)

  is_cd8 <- grepl("^CD8\\+ T", refined_guarded) |
            refined_guarded %in% c("CD8+ T cells","CD8+ T naive",
                                   "CD8+ T memory","CD8+ T effector",
                                   "CD8+ T effector memory")
  cd8_idx <- which(is_cd8)
  if (length(cd8_idx)) {
    thr <- stats::quantile(ex_score[cd8_idx], 0.80, na.rm = TRUE)
    term_idx <- cd8_idx[ ex_score[cd8_idx] > thr & ex_score[cd8_idx] > 0.5 ]
    if (length(term_idx)) refined_guarded[term_idx] <- "CD8+ T exhausted"
    message("Exhausted CD8 re-labeled: ", length(term_idx))
  } else {
    message("No CD8-like cells detected; skipping exhaustion.")
  }
} else {
  message("No exhaustion markers in data; skipping exhaustion.")
}

# 3) Fix Monocyte ↔ DC conflicts using Monaco fine labels
monaco_fine <- cd45_cells$cell_type_monaco_fine
lab <- refined_guarded

is_mono <- grepl("^Monocytes", lab)
is_dc_by_monaco <- grepl("dendritic", monaco_fine, TRUE)
conflict <- which(is_mono & is_dc_by_monaco)

to_mDC <- conflict[grepl("myeloid|conventional|cDC|DC1|DC2", monaco_fine[conflict], TRUE)]
to_pDC <- conflict[grepl("plasmacytoid|pDC",                 monaco_fine[conflict], TRUE)]
ambig  <- setdiff(conflict, c(to_mDC, to_pDC))

lab[to_mDC] <- "mDC"
lab[to_pDC] <- "pDC"
if (length(ambig)) lab[ambig] <- "Dendritic cells"

message("Resolved Monocyte↔DC conflicts: ", length(conflict),
        "  (mDC:", length(to_mDC), ", pDC:", length(to_pDC), ", DC:", length(ambig), ")")

# 4) Split generic "T cells" into CD4 vs CD8 by CD4 vs CD8A/B
genes <- rownames(expr_data)
t_idx <- which(lab == "T cells")
if (length(t_idx)) {
  cd4  <- if ("CD4"  %in% genes) as.numeric(expr_data["CD4",  t_idx]) else rep(0, length(t_idx))
  cd8A <- if ("CD8A" %in% genes) as.numeric(expr_data["CD8A", t_idx]) else rep(0, length(t_idx))
  cd8B <- if ("CD8B" %in% genes) as.numeric(expr_data["CD8B", t_idx]) else rep(0, length(t_idx))
  cd8  <- ifelse(cd8B > 0, (cd8A + cd8B)/2, cd8A)

  assign_cd4 <- t_idx[ cd4 >  cd8 * 1.2 ]
  assign_cd8 <- t_idx[ cd8 >= cd4 * 1.2 ]  # >= to break ties toward CD8
  lab[assign_cd4] <- "CD4+ T cells"
  lab[assign_cd8] <- "CD8+ T cells"
  message("Resolved generic T cells -> CD4: ", length(assign_cd4), " | CD8: ", length(assign_cd8))
}

# 5) Tidy residual "Monocytes" into classical / non-classical via CD14 vs FCGR3A
mono_idx <- which(lab == "Monocytes")
if (length(mono_idx)) {
  cd14   <- if ("CD14"   %in% genes) as.numeric(expr_data["CD14",   mono_idx]) else rep(0, length(mono_idx))
  fcgr3a <- if ("FCGR3A" %in% genes) as.numeric(expr_data["FCGR3A", mono_idx]) else rep(0, length(mono_idx))
  lab[mono_idx[cd14 >= fcgr3a]] <- "Monocytes classical"
  lab[mono_idx[fcgr3a >  cd14 ]] <- "Monocytes non-classical"
  message("Resolved residual 'Monocytes' -> classical/non-classical: ", length(mono_idx))
}

# 6) Finalize and save
refined_cell_types_final <- lab
names(refined_cell_types_final) <- colnames(cd45_cells)

# Summary table
tab_final <- sort(table(refined_cell_types_final), decreasing = TRUE)
print(tab_final)

# Save labels + comparison
output_dir <- "/content/drive/MyDrive/BayesPrism/processed_data"
if (!dir.exists(output_dir)) dir.create(output_dir, recursive = TRUE)

saveRDS(refined_cell_types_final,
        file.path(output_dir, "multi_reference_refined_cell_types_patched.rds"))

comparison_df <- data.frame(
  cell_barcode = colnames(cd45_cells),
  monaco_only  = refined_cell_types_monaco,
  dice_fine    = cd45_cells$cell_type_dice_fine,
  final_patched= refined_cell_types_final,
  stringsAsFactors = FALSE
)
write.csv(comparison_df,
          file.path(output_dir, "multi_reference_refinement_comparison_patched.csv"),
          row.names = FALSE)

cat("\nSaved:\n - multi_reference_refined_cell_types_patched.rds\n",
    " - multi_reference_refinement_comparison_patched.csv\n", sep = "")


In [ ]:
%%R
# Promote Macrophage/TAM from Monocytes
suppressPackageStartupMessages({ library(Seurat); library(dplyr) })

stopifnot(exists("cd45_cells"), exists("refined_cell_types_final"))
out_dir <- "/content/drive/MyDrive/BayesPrism/processed_data"
if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)

labels0 <- refined_cell_types_final
names(labels0) <- colnames(cd45_cells)

# --- Marker panels ---
mono_sig  <- c("LYZ","S100A8","S100A9","FCN1","MS4A7","CTSS","VCAN","TYROBP","CCR2","LST1","FCGR3A","CX3CR1")
macro_sig <- c("CD68","MSR1","CD163","MRC1","C1QA","C1QB","C1QC","APOE","MARCO","SIGLEC1","TREM2","SPP1")
# guards
dc_sig    <- c("FLT3","IRF8","CLEC9A","XCR1","CD1C","FCER1A","LAMP3","CCR7")
neut_sig  <- c("CXCR2","FCGR3B","MMP8","CSF3R","ELANE","MPO")

expr <- GetAssayData(cd45_cells, slot="data")
avail <- function(s) intersect(s, rownames(expr))
mono_sig  <- avail(mono_sig);   macro_sig <- avail(macro_sig)
dc_sig    <- avail(dc_sig);     neut_sig  <- avail(neut_sig)

if (length(mono_sig) < 3 || length(macro_sig) < 3) {
  stop(sprintf("Not enough mono/macro markers (mono=%d, macro=%d). Tweak lists.",
               length(mono_sig), length(macro_sig)))
}

# Program scores
cd45_cells <- AddModuleScore(cd45_cells, features=list(mono_sig),  name="MonoLin")
cd45_cells <- AddModuleScore(cd45_cells, features=list(macro_sig), name="MacroLin")
if (length(dc_sig)   >= 3) cd45_cells <- AddModuleScore(cd45_cells, features=list(dc_sig),   name="DCLin")
if (length(neut_sig) >= 3) cd45_cells <- AddModuleScore(cd45_cells, features=list(neut_sig), name="NeutLin")

md <- cd45_cells@meta.data
md$MonoLin  <- md$MonoLin1
md$MacroLin <- md$MacroLin1
md$Delta_MM <- md$MacroLin - md$MonoLin   # >0 favors Macrophage/TAM

# Candidates: current Monocytes only
cand <- which(grepl("^Monocytes( |$)|^Monocytes classical$|^Monocytes non-classical$", labels0))
labels1 <- labels0
if (length(cand)) {
  # z-score within candidates
  z <- rep(NA_real_, ncol(cd45_cells)); z[cand] <- as.numeric(scale(md$Delta_MM[cand]))

  # Guard rails: block flips if DC/Neut very high (top 10% globally)
  guard <- rep(FALSE, length(cand))
  if ("DCLin1" %in% names(md))   guard <- guard | (md$DCLin1[cand]   > quantile(md$DCLin1,   0.90, na.rm=TRUE))
  if ("NeutLin1" %in% names(md)) guard <- guard | (md$NeutLin1[cand] > quantile(md$NeutLin1, 0.90, na.rm=TRUE))

  # Require ≥3 macrophage hallmark genes above gene-median
  macro_hallmarks <- avail(c("C1QA","C1QB","C1QC","APOE","TREM2","SPP1","SIGLEC1","MARCO","MRC1","CD163"))
  hall_ok <- rep(TRUE, length(cand))
  if (length(macro_hallmarks) >= 3) {
    med <- apply(expr[macro_hallmarks, , drop=FALSE], 1, median, na.rm=TRUE)
    above_med <- sweep(expr[macro_hallmarks, , drop=FALSE], 1, med, FUN=">") * 1
    hall_counts <- colSums(above_med, na.rm=TRUE)
    hall_ok <- hall_counts[colnames(expr)[cand]] >= 3
  }

  # Threshold (conservative)
  z_hi <- 1.8
  flip_to_macro <- cand[ (z[cand] >= z_hi) & !guard & hall_ok ]

  labels1[flip_to_macro] <- "Macrophage"

  # change log
  changes <- data.frame(
    cell_barcode = names(labels0)[flip_to_macro],
    old_label    = labels0[flip_to_macro],
    new_label    = labels1[flip_to_macro],
    Delta_MM     = md$Delta_MM[flip_to_macro],
    z_Delta_MM   = z[flip_to_macro],
    stringsAsFactors = FALSE
  )
  write.csv(changes,
            file.path(out_dir, "refinement_changes_monocyte_to_macrophage.csv"),
            row.names = FALSE)

  message("Promoted to Macrophage/TAM: ", length(flip_to_macro), " cells")
} else {
  message("No monocyte candidates found; skipping Macrophage promotion.")
}

# M1/M2 polarity scores (QC only; not labels)
M1 <- avail(c("STAT1","IRF1","GBP1","CXCL9","CXCL10","SOCS3","IL12B"))
M2 <- avail(c("MRC1","CD163","MSR1","CCL18","MERTK","MAF","PPARG"))
if (length(M1) >= 2) cd45_cells <- AddModuleScore(cd45_cells, features=list(M1), name="M1pol")
if (length(M2) >= 2) cd45_cells <- AddModuleScore(cd45_cells, features=list(M2), name="M2pol")

# Update global label
refined_cell_types_final <- labels1
names(refined_cell_types_final) <- colnames(cd45_cells)
saveRDS(refined_cell_types_final,
        file.path(out_dir, "refined_cell_types_with_macrophage.rds"))

if (all(c("M1pol1","M2pol1") %in% names(cd45_cells@meta.data))) {
  pol_df <- data.frame(
    cell_barcode = colnames(cd45_cells),
    label        = refined_cell_types_final,
    M1_score     = cd45_cells$M1pol1,
    M2_score     = cd45_cells$M2pol1,
    M2_minus_M1  = cd45_cells$M2pol1 - cd45_cells$M1pol1,
    stringsAsFactors = FALSE
  )
  write.csv(pol_df, file.path(out_dir, "macrophage_polarization_scores.csv"), row.names=FALSE)
}

cat("\nCounts (key myeloid):\n")
print(sort(table(refined_cell_types_final[
  grepl("^Monocytes|^Macrophage|^mDC$|^pDC$|^Dendritic", refined_cell_types_final)
]), decreasing=TRUE))
cat("\nSaved:\n - refined_cell_types_with_macrophage.rds\n",
    "- refinement_changes_monocyte_to_macrophage.csv (if any)\n",
    if (all(c('M1pol1','M2pol1') %in% names(cd45_cells@meta.data)))
      "- macrophage_polarization_scores.csv\n" else "", sep="")


In [ ]:
%%R
# Adaptive, guarded Macrophage/TAM promotion from Monocytes
# - Targets a modest TAM size while keeping strict guard rails
# - Requires ≥ target_min cells AND ≥ min_donors donors
# - Caps TAM at ≤ max_prop of all Monocytes
suppressPackageStartupMessages({ library(Seurat); library(dplyr) })

stopifnot(exists("cd45_cells"), exists("refined_cell_types_final"))
out_dir <- "/content/drive/MyDrive/BayesPrism/processed_data"
if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)

labels0 <- refined_cell_types_final
names(labels0) <- colnames(cd45_cells)

target_min  <- 40     # aim for at least this many TAM cells
max_prop    <- 0.15   # don't exceed 20% of all monocytes
z_floor     <- 1.0    # don't promote below this z-score (macro - mono)
min_donors  <- 2      # require TAM across at least this many donors

# ---- ensure program scores exist (MonoLin1 / MacroLin1) ----
expr <- GetAssayData(cd45_cells, slot="data")
avail <- function(s) intersect(s, rownames(expr))
mono_sig  <- avail(c("LYZ","S100A8","S100A9","FCN1","MS4A7","CTSS","VCAN","TYROBP","CCR2","LST1","FCGR3A","CX3CR1"))
macro_sig <- avail(c("CD68","MSR1","CD163","MRC1","C1QA","C1QB","C1QC","APOE","MARCO","SIGLEC1","TREM2","SPP1"))
dc_sig    <- avail(c("FLT3","IRF8","CLEC9A","XCR1","CD1C","FCER1A","LAMP3","CCR7"))
neut_sig  <- avail(c("CXCR2","FCGR3B","MMP8","CSF3R","ELANE","MPO"))

if (!all(c("MonoLin1","MacroLin1") %in% names(cd45_cells@meta.data))) {
  if (length(mono_sig) < 3 || length(macro_sig) < 3)
    stop("Not enough mono/macro markers to score. Adjust lists.")
  cd45_cells <- AddModuleScore(cd45_cells, features=list(mono_sig),  name="MonoLin")
  cd45_cells <- AddModuleScore(cd45_cells, features=list(macro_sig), name="MacroLin")
}
if (length(dc_sig)   >= 3 && !"DCLin1"   %in% names(cd45_cells@meta.data)) cd45_cells <- AddModuleScore(cd45_cells, features=list(dc_sig),   name="DCLin")
if (length(neut_sig) >= 3 && !"NeutLin1" %in% names(cd45_cells@meta.data)) cd45_cells <- AddModuleScore(cd45_cells, features=list(neut_sig), name="NeutLin")

md <- cd45_cells@meta.data
md$Delta_MM <- md$MacroLin1 - md$MonoLin1

# ---- candidates: current monocytes only ----
cand <- which(grepl("^Monocytes( |$)|^Monocytes classical$|^Monocytes non-classical$", labels0))
n_mono <- length(cand)

if (n_mono == 0) {
  message("No monocytes present; nothing to promote.")
  refined_cell_types_final <- labels0
} else {
  # z-score within candidates
  z <- rep(NA_real_, ncol(cd45_cells)); z[cand] <- as.numeric(scale(md$Delta_MM[cand]))

  # guard rails: block if DC/Neut very high (top 10% globally)
  guard <- rep(FALSE, length(cand))
  if ("DCLin1" %in% names(md))   guard <- guard | (md$DCLin1[cand]   > quantile(md$DCLin1,   0.90, na.rm=TRUE))
  if ("NeutLin1" %in% names(md)) guard <- guard | (md$NeutLin1[cand] > quantile(md$NeutLin1, 0.90, na.rm=TRUE))

  # hallmark requirement: ≥3 macrophage genes above gene-median
  macro_hallmarks <- avail(c("C1QA","C1QB","C1QC","APOE","TREM2","SPP1","SIGLEC1","MARCO","MRC1","CD163"))
  hall_ok <- rep(TRUE, length(cand))
  if (length(macro_hallmarks) >= 3) {
    med <- apply(expr[macro_hallmarks, , drop=FALSE], 1, median, na.rm=TRUE)
    above_med <- sweep(expr[macro_hallmarks, , drop=FALSE], 1, med, FUN=">") * 1
    hall_counts <- colSums(above_med, na.rm=TRUE)
    hall_ok <- hall_counts[colnames(expr)[cand]] >= 3
  }

  # eligibility mask
  elig <- (!guard) & hall_ok & (z[cand] >= z_floor)
  elig_idx <- cand[elig]

  # donor coverage info
  pick <- c("sample_id","Sample","patient","Patient","orig.ident","orig.ident2","donor","Donor")
  scol <- pick[pick %in% colnames(cd45_cells@meta.data)][1]
  donors <- if (!is.na(scol)) cd45_cells@meta.data[[scol]] else rep("NA", ncol(cd45_cells))
  names(donors) <- colnames(cd45_cells)

  # choose how many to promote
  k_target <- max(target_min, round(0.10 * n_mono))              # aim ~10% or >= target_min
  k_cap    <- floor(max_prop * n_mono)                           # never exceed cap
  k        <- min(k_target, k_cap, length(elig_idx))

  labels1 <- labels0
  if (k >= target_min) {
    ord <- elig_idx[order(z[elig_idx], decreasing=TRUE)]
    sel <- ord[seq_len(k)]

    # donor coverage check
    n_donors <- length(unique(donors[sel]))
    if (n_donors >= min_donors) {
      labels1[sel] <- "Macrophage"
      promoted <- sel
      reason <- sprintf("Promoted %d cells (%.1f%% of monocytes) across %d donors.",
                        length(sel), 100*length(sel)/n_mono, n_donors)
    } else {
      promoted <- integer(0)
      reason <- sprintf("Donor coverage too low for TAM (got %d, need %d). Keeping previous labels.", n_donors, min_donors)
    }
  } else {
    promoted <- integer(0)
    reason <- sprintf("Only %d eligible ≥ z_floor=%.1f (target %d, cap %d). Keeping previous labels.",
                      length(elig_idx), z_floor, k_target, k_cap)
  }

  if (length(promoted) == 0) {
    refined_cell_types_final <- labels0  # fallback
  } else {
    refined_cell_types_final <- labels1
    # log changes
    changes <- data.frame(
      cell_barcode = names(labels0)[promoted],
      old_label    = labels0[promoted],
      new_label    = refined_cell_types_final[promoted],
      Delta_MM     = md$Delta_MM[promoted],
      z_Delta_MM   = z[promoted],
      donor        = donors[promoted],
      stringsAsFactors = FALSE
    )
    write.csv(changes, file.path(out_dir, "refinement_changes_monocyte_to_macrophage_ADAPTIVE.csv"), row.names=FALSE)
  }

  message(reason)
}

# Save
saveRDS(refined_cell_types_final, file.path(out_dir, "refined_cell_types_with_macrophage_ADAPTIVE.rds"))
cat("\nMyeloid counts after adaptive TAM:\n")
print(sort(table(refined_cell_types_final[
  grepl("^Monocytes|^Macrophage|^mDC$|^pDC$|^Dendritic", refined_cell_types_final)
]), decreasing=TRUE))
cat("\nSaved:\n - refined_cell_types_with_macrophage_ADAPTIVE.rds\n",
    "- refinement_changes_monocyte_to_macrophage_ADAPTIVE.csv (if promoted)\n", sep="")


In [ ]:
%%R
# Macrophage misclassification audit
suppressPackageStartupMessages({ library(Seurat); library(Matrix); library(dplyr) })

stopifnot(exists("cd45_cells"), exists("refined_cell_types_final"))
labels0 <- refined_cell_types_final
names(labels0) <- colnames(cd45_cells)

do_relabel <- FALSE      # <- set TRUE to actually relabel
z_cut      <- 1.2        # loosened vs conservative 1.8
q_macro    <- 0.85       # require MacroLin in top 15% among monocytes
min_halls  <- 3          # hallmarks needed
guard_topq <- 0.90       # block top 10% DC/Neut

# ---- marker sets ----
expr <- GetAssayData(cd45_cells, slot="data")
avail <- function(s) intersect(s, rownames(expr))

mono_sig  <- avail(c("LYZ","S100A8","S100A9","FCN1","MS4A7","CTSS","VCAN","TYROBP","CCR2","LST1","FCGR3A","CX3CR1"))
macro_sig <- avail(c("CD68","MSR1","CD163","MRC1","C1QA","C1QB","C1QC","APOE","MARCO","SIGLEC1","TREM2","SPP1"))
dc_sig    <- avail(c("FLT3","IRF8","CLEC9A","XCR1","CD1C","FCER1A","LAMP3","CCR7"))
neut_sig  <- avail(c("CXCR2","FCGR3B","MMP8","CSF3R","ELANE","MPO"))
hallmarks <- avail(c("C1QA","C1QB","C1QC","APOE","TREM2","SPP1","SIGLEC1","MARCO","MRC1","CD163"))

if (!all(c("MonoLin1","MacroLin1") %in% colnames(cd45_cells@meta.data))) {
  if (length(mono_sig) < 3 || length(macro_sig) < 3) stop("Not enough mono/macro markers.")
  cd45_cells <- AddModuleScore(cd45_cells, features=list(mono_sig),  name="MonoLin")
  cd45_cells <- AddModuleScore(cd45_cells, features=list(macro_sig), name="MacroLin")
}
if (length(dc_sig)   >= 3 && !"DCLin1"   %in% colnames(cd45_cells@meta.data)) cd45_cells <- AddModuleScore(cd45_cells, features=list(dc_sig),   name="DCLin")
if (length(neut_sig) >= 3 && !"NeutLin1" %in% colnames(cd45_cells@meta.data)) cd45_cells <- AddModuleScore(cd45_cells, features=list(neut_sig), name="NeutLin")

md <- cd45_cells@meta.data
md$Delta_MM <- md$MacroLin1 - md$MonoLin1

# Monocyte pool
mono_idx <- which(grepl("^Monocytes( |$)|^Monocytes classical$|^Monocytes non-classical$", labels0))
if (!length(mono_idx)) {
  message("No monocytes present; nothing to audit.")
} else {
  # Z within monocytes
  z <- rep(NA_real_, ncol(cd45_cells)); z[mono_idx] <- as.numeric(scale(md$Delta_MM[mono_idx]))

  # DC/Neut guards (global top 10%)
  dc_hi   <- if ("DCLin1"   %in% names(md)) md$DCLin1   > quantile(md$DCLin1,   guard_topq, na.rm=TRUE) else rep(FALSE, ncol(cd45_cells))
  neut_hi <- if ("NeutLin1" %in% names(md)) md$NeutLin1 > quantile(md$NeutLin1, guard_topq, na.rm=TRUE) else rep(FALSE, ncol(cd45_cells))

  # Hallmark count above gene-wide medians
  hall_ok <- rep(TRUE, ncol(cd45_cells))
  if (length(hallmarks) >= min_halls) {
    med <- apply(expr[hallmarks, , drop=FALSE], 1, median, na.rm=TRUE)
    above_med <- sweep(expr[hallmarks, , drop=FALSE], 1, med, FUN=">") * 1
    hcounts <- colSums(above_med, na.rm=TRUE)
    hall_ok <- hcounts >= min_halls
  }

  # Macro intensity quantile within monocytes
  macro_q <- rep(NA_real_, ncol(cd45_cells))
  if (length(mono_idx) >= 20) {
    qvals <- ecdf(md$MacroLin1[mono_idx])
    macro_q <- qvals(md$MacroLin1)
  } else {
    macro_q <- rep(1, ncol(cd45_cells))
  }

  # Candidate rule: strong macro bias, hallmark evidence, not DC/Neut-like, macro high quantile
  cand <- mono_idx[ z[mono_idx] >= z_cut & hall_ok[mono_idx] &
                    !dc_hi[mono_idx] & !neut_hi[mono_idx] &
                    macro_q[mono_idx] >= q_macro ]

  # Report
  cat("\nAudit: macrophage-like monocytes found:", length(cand), "cells\n")
  if (length(cand)) {
    # show top 25 by z
    top <- cand[order(z[cand], decreasing=TRUE)]
    head_tbl <- data.frame(
      cell_barcode = colnames(cd45_cells)[head(top, 25)],
      label        = labels0[head(top,25)],
      Delta_MM     = md$Delta_MM[head(top,25)],
      z_Delta_MM   = z[head(top,25)],
      MacroLin     = md$MacroLin1[head(top,25)],
      MonoLin      = md$MonoLin1[head(top,25)]
    )
    print(head_tbl, row.names=FALSE)
  }

  # Optional relabel
  labels1 <- labels0
  if (do_relabel && length(cand)) {
    labels1[cand] <- "Macrophage"
    refined_cell_types_final <- labels1
    cat("\nRelabeled", length(cand), "monocytes to Macrophage (TAM).\n")
  } else {
    refined_cell_types_final <- labels0
  }

  # Save a candidate CSV (for review)
  out_dir <- "/content/drive/MyDrive/BayesPrism/processed_data"
  if (!dir.exists(out_dir)) dir.create(out_dir, recursive=TRUE)
  cand_df <- data.frame(
    cell_barcode = colnames(cd45_cells)[cand],
    old_label    = labels0[cand],
    Delta_MM     = md$Delta_MM[cand],
    z_Delta_MM   = z[cand],
    MacroLin     = md$MacroLin1[cand],
    MonoLin      = md$MonoLin1[cand],
    stringsAsFactors = FALSE
  )
  write.csv(cand_df, file.path(out_dir, "macrophage_misclassification_candidates.csv"), row.names=FALSE)

  # Print myeloid summary
  cat("\nMyeloid counts (after audit",
      if (do_relabel) "+ rescue" else "",
      "):\n", sep="")
  print(sort(table(refined_cell_types_final[
    grepl("^Monocytes|^Macrophage|^mDC$|^pDC$|^Dendritic", refined_cell_types_final)
  ]), decreasing=TRUE))


  if (do_relabel) {
    saveRDS(refined_cell_types_final,
            file.path(out_dir, "refined_cell_types_with_macrophage_AUDIT_RESCUE.rds"))
    cat("\nSaved relabeled file: refined_cell_types_with_macrophage_AUDIT_RESCUE.rds\n")
  }
  cat("Candidate list saved: macrophage_misclassification_candidates.csv\n")
}


In [ ]:
%%R
# 1) Cell lost
stopifnot(ncol(cd45_cells) == length(refined_cell_types_final))
stopifnot(identical(colnames(cd45_cells), names(refined_cell_types_final)))
cat("OK: same cells before/after (", ncol(cd45_cells), " cells )\n", sep="")

# 2) Myeloid dendritic cells
monaco_dc <- cd45_cells$cell_type_monaco_fine == "Myeloid dendritic cells"
cat("Monaco mDC count:", sum(monaco_dc), "\n")
print(sort(table(refined_cell_types_final[monaco_dc]), decreasing=TRUE))

# 3) NK cells (DICE often overcalls NK)
dice_nk <- cd45_cells$cell_type_dice_fine == "NK cells"
cat("\nDICE NK count:", sum(dice_nk), "\n")
print(sort(table(refined_cell_types_final[dice_nk]), decreasing=TRUE))

# 4) Top-20 MonacoFine → Refined pairs (confusion view)
df <- as.data.frame(table(MonacoFine=cd45_cells$cell_type_monaco_fine,
                          Refined=refined_cell_types_final))
df <- df[df$Freq > 0, ]
df <- df[order(-df$Freq), ]
cat("\nTop Monaco→Refined pairs:\n")
print(head(df, 20), row.names=FALSE)


In [ ]:
%%R
expr <- GetAssayData(cd45_cells, slot="data")
genes <- rownames(expr)

gd_markers <- intersect(c("TRDC","TRGC1","TRGC2","TRGV9","TRDV1","TRDV2","TRDV3"), genes)
ab_markers <- intersect(c("TRAC","TRBC1","TRBC2","CD3E"), genes)

if (length(gd_markers) >= 2) {
  gd_score <- colMeans(expr[gd_markers, , drop=FALSE])
  ab_score <- if (length(ab_markers)) colMeans(expr[ab_markers, , drop=FALSE]) else rep(0, ncol(cd45_cells))

  t_idx <- which(grepl("(^T cells$)|(^CD[48]\\+ T )", refined_cell_types_final))
  thr_gd <- quantile(gd_score[t_idx], 0.90, na.rm=TRUE)

  gd_idx <- t_idx[ gd_score[t_idx] >= thr_gd & gd_score[t_idx] > 0.3 &
                   ab_score[t_idx] <= quantile(ab_score[t_idx], 0.50, na.rm=TRUE) ]

  if (length(gd_idx)) {
    refined_cell_types_final[gd_idx] <- "T cells gamma delta"
    cat("Tagged gamma-delta T cells:", length(gd_idx), "\n")
  }
}


In [ ]:
%%R
# Save final labels and export ONLY the FINE BayesPrism reference

suppressPackageStartupMessages({ library(Seurat); library(Matrix) })

stopifnot(exists("cd45_cells"), exists("refined_cell_types_final"))

names(refined_cell_types_final) <- colnames(cd45_cells)

out_dir <- "/content/drive/MyDrive/BayesPrism/processed_data"
if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)

# --- 1) Save labels and Seurat with labels embedded ---
saveRDS(refined_cell_types_final,
        file.path(out_dir, "refined_cell_types_final.rds"))
write.csv(
  data.frame(cell_barcode = names(refined_cell_types_final),
             final_label  = unname(refined_cell_types_final)),
  file.path(out_dir, "refined_cell_types_final.csv"),
  row.names = FALSE
)
cd45_cells$refined_final <- refined_cell_types_final
saveRDS(cd45_cells,
        file.path(out_dir, "cd45_seurat_object_with_refined_final.rds"))

cat("Saved:\n- refined_cell_types_final.rds\n- refined_cell_types_final.csv\n- cd45_seurat_object_with_refined_final.rds\n")

# --- 2) Export FINE reference ---
min_cells_fine <- 20

counts <- GetAssayData(cd45_cells, slot = "counts")
lab    <- refined_cell_types_final

tbl <- sort(table(lab), decreasing = TRUE)
keep_types <- names(tbl)[tbl >= min_cells_fine]
cells_keep <- names(lab)[lab %in% keep_types]

# Sanity
stopifnot(length(cells_keep) > 0)

ref_counts <- counts[, cells_keep, drop = FALSE]
# light gene filter: remove very rare genes
g_tot <- Matrix::rowSums(ref_counts)
ref_counts <- ref_counts[g_tot >= 10, , drop = FALSE]
ref_types  <- lab[cells_keep]

# align
stopifnot(identical(colnames(ref_counts), names(ref_types)))

# Save FINE reference (RDS + MM + CSV)
saveRDS(ref_counts, file.path(out_dir, "bp_ref_counts_sparse_FINE.rds"))
saveRDS(ref_types,  file.path(out_dir, "bp_ref_cell_types_FINE.rds"))

Matrix::writeMM(ref_counts, file.path(out_dir, "bp_ref_counts_FINE.mtx"))
write.table(rownames(ref_counts), file.path(out_dir, "bp_ref_genes_FINE.txt"),
            quote = FALSE, row.names = FALSE, col.names = FALSE)
write.table(colnames(ref_counts), file.path(out_dir, "bp_ref_barcodes_FINE.txt"),
            quote = FALSE, row.names = FALSE, col.names = FALSE)
write.csv(data.frame(cell_barcode = names(ref_types),
                     cell_type    = unname(ref_types)),
          file.path(out_dir, "bp_ref_cell_types_FINE.csv"),
          row.names = FALSE)

coverage <- sum(tbl[keep_types]) / sum(tbl)
cat("\n[FINE] kept types (≥", min_cells_fine, "): ",
    paste(keep_types, collapse = ", "), "\n", sep = "")
cat(sprintf("[FINE] reference coverage: %.1f%% of cells\n", 100*coverage))

if (length(setdiff(names(tbl), keep_types))) {
  cat("[FINE] excluded (too small): ",
      paste(setdiff(names(tbl), keep_types), collapse = ", "), "\n", sep = "")
}

cat("\nFINE reference saved:\n",
    " - bp_ref_counts_sparse_FINE.rds\n",
    " - bp_ref_cell_types_FINE.rds\n",
    " - bp_ref_counts_FINE.mtx / bp_ref_genes_FINE.txt / bp_ref_barcodes_FINE.txt\n",
    " - bp_ref_cell_types_FINE.csv\n", sep = "")


In [ ]:
%%R
# Add TAM
suppressPackageStartupMessages({ library(Seurat); library(Matrix) })

stopifnot(exists("cd45_cells"), exists("refined_cell_types_final"))
names(refined_cell_types_final) <- colnames(cd45_cells)

out_dir <- "/content/drive/MyDrive/BayesPrism/processed_data"
if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)

# ---------- 1) Marker sets ----------
mono_sig  <- c("LYZ","S100A8","S100A9","FCN1","MS4A7","CTSS","VCAN","TYROBP","CCR2","LST1","FCGR3A","CX3CR1")
macro_sig <- c("CD68","MSR1","CD163","MRC1","C1QA","C1QB","C1QC","APOE","MARCO","SIGLEC1","TREM2","SPP1")
tam_hallmarks <- c("C1QA","C1QB","C1QC","APOE","TREM2","SPP1","SIGLEC1","MARCO","MRC1","CD163")

# guard rails to avoid DC / neut contamination
dc_sig    <- c("FLT3","IRF8","CLEC9A","XCR1","CD1C","FCER1A","LAMP3","CCR7")
neut_sig  <- c("CXCR2","FCGR3B","MMP8","CSF3R","ELANE","MPO")

expr <- GetAssayData(cd45_cells, slot = "data")
avail <- function(s) intersect(s, rownames(expr))

mono_sig  <- avail(mono_sig);    macro_sig <- avail(macro_sig)
dc_sig    <- avail(dc_sig);      neut_sig  <- avail(neut_sig)
tam_hallmarks <- avail(tam_hallmarks)

if (length(macro_sig) < 3 || length(mono_sig) < 3) {
  stop("Not enough markers to compute macrophage/monocyte programs.")
}

# ---------- 2) Program scores ----------
cd45_cells <- AddModuleScore(cd45_cells, features = list(mono_sig),  name = "MonoLin")
cd45_cells <- AddModuleScore(cd45_cells, features = list(macro_sig), name = "MacroLin")
if (length(dc_sig)   >= 3) cd45_cells <- AddModuleScore(cd45_cells, features = list(dc_sig),   name = "DCLin")
if (length(neut_sig) >= 3) cd45_cells <- AddModuleScore(cd45_cells, features = list(neut_sig), name = "NeutLin")

md <- cd45_cells@meta.data
md$Delta_MM <- md$MacroLin1 - md$MonoLin1

# ---------- 3) Candidate pool: monocyte lineage (+ existing Macrophage) ----------
lab0 <- refined_cell_types_final
cand <- which(lab0 %in% c("Monocytes","Monocytes classical","Monocytes non-classical","Macrophage"))
if (!length(cand)) stop("No monocyte/macrophage candidates found.")

# Exclude any Monaco-DC-labeled cells (extra safety)
if ("cell_type_monaco_fine" %in% colnames(cd45_cells@meta.data)) {
  is_dc_by_monaco <- grepl("dendritic", cd45_cells$cell_type_monaco_fine, ignore.case=TRUE)
  cand <- setdiff(cand, which(is_dc_by_monaco))
}

# ---------- 4) Hallmark gate & guard rails ----------
hall_ok <- rep(TRUE, ncol(cd45_cells))
if (length(tam_hallmarks) >= 3) {
  med <- apply(expr[tam_hallmarks, , drop=FALSE], 1, median, na.rm=TRUE)
  above_med <- sweep(expr[tam_hallmarks, , drop=FALSE], 1, med, `>`) * 1
  hall_counts <- colSums(above_med, na.rm=TRUE)
  hall_ok <- hall_counts >= 3
}
guard <- rep(FALSE, ncol(cd45_cells))
if ("DCLin1" %in% names(md))   guard <- guard | (md$DCLin1 >  quantile(md$DCLin1,  0.90, na.rm=TRUE))
if ("NeutLin1" %in% names(md)) guard <- guard | (md$NeutLin1 > quantile(md$NeutLin1, 0.90, na.rm=TRUE))

# ---------- 5) Adaptive Z-thresholding to reach min TAM size ----------
z_all <- as.numeric(scale(md$Delta_MM))          # global z to be conservative
z <- z_all[cand]

target_min <- 30        # desired minimum TAM size
z_start    <- 1.8
z_min      <- 1.0
step       <- 0.1

pick <- integer(0)
z_thr <- z_start
repeat {
  pick <- cand[ (z >= z_thr) & !guard[cand] & hall_ok[cand] ]
  if (length(pick) >= target_min || z_thr <= z_min) break
  z_thr <- z_thr - step
}

# If small, fall back to top-N by Delta_MM among candidates
if (length(pick) < target_min) {
  ord <- cand[order(md$Delta_MM[cand], decreasing=TRUE)]
  pick <- unique(c(pick, head(ord, target_min)))
}

cat("TAM selected:", length(pick), "cells (z_thr ~", round(z_thr,2), ")\n")

# ---------- 6) Apply TAM label ----------
lab1 <- lab0
lab1[lab1 == "Macrophage"] <- "TAM"
lab1[pick] <- "TAM"

# ---------- 7) Save updated labels + Seurat ----------
saveRDS(lab1, file.path(out_dir, "refined_cell_types_final_WITH_TAM.rds"))
write.csv(data.frame(cell_barcode = names(lab1), final_label = unname(lab1)),
          file.path(out_dir, "refined_cell_types_final_WITH_TAM.csv"),
          row.names = FALSE)

cd45_cells$refined_final <- lab1
saveRDS(cd45_cells, file.path(out_dir, "cd45_seurat_object_with_TAM.rds"))

cat("Saved:\n- refined_cell_types_final_WITH_TAM.rds\n- refined_cell_types_final_WITH_TAM.csv\n- cd45_seurat_object_with_TAM.rds\n")

# ---------- 8) Counts snapshot ----------
print(sort(table(lab1[grepl("TAM|Monocytes|mDC|pDC|Dendritic|NK|B cells|CD8|CD4|MAIT", lab1)]), decreasing=TRUE))

# ---------- 9) Export FINE reference with TAM ----------
min_cells_fine <- 20

counts <- GetAssayData(cd45_cells, slot="counts")
tbl <- table(lab1)
keep_types <- names(tbl)[tbl >= min_cells_fine]
cells_keep <- names(lab1)[lab1 %in% keep_types]

ref_counts <- counts[, cells_keep, drop=FALSE]
g_tot <- Matrix::rowSums(ref_counts)
ref_counts <- ref_counts[g_tot >= 10, , drop=FALSE]
ref_types  <- lab1[cells_keep]
stopifnot(identical(colnames(ref_counts), names(ref_types)))

saveRDS(ref_counts, file.path(out_dir, "bp_ref_counts_sparse_FINE.rds"))
saveRDS(ref_types,  file.path(out_dir, "bp_ref_cell_types_FINE.rds"))
Matrix::writeMM(ref_counts, file.path(out_dir, "bp_ref_counts_FINE.mtx"))
write.table(rownames(ref_counts), file.path(out_dir, "bp_ref_genes_FINE.txt"),
            quote=FALSE, row.names=FALSE, col.names=FALSE)
write.table(colnames(ref_counts), file.path(out_dir, "bp_ref_barcodes_FINE.txt"),
            quote=FALSE, row.names=FALSE, col.names=FALSE)
write.csv(data.frame(cell_barcode = names(ref_types),
                     cell_type    = unname(ref_types)),
          file.path(out_dir, "bp_ref_cell_types_FINE.csv"),
          row.names = FALSE)

coverage <- sum(tbl[keep_types]) / sum(tbl)
cat("\n[FINE] kept (≥", min_cells_fine, "): ", paste(keep_types, collapse=", "),
    "\n[FINE] coverage: ", sprintf("%.1f%%", 100*coverage), "\n", sep="")


In [ ]:
%%R
# COMPREHENSIVE BAYESPRISM DECONVOLUTION
# Using refined immune cell subtypes

# Load required libraries
library(org.Hs.eg.db)
library(AnnotationDbi)
library(BayesPrism)

# Set file paths
output_dir <- "/content/drive/MyDrive/BayesPrism/processed_data"
results_dir <- "/content/drive/MyDrive/BayesPrism/results_annotation"


In [ ]:
%%R
# FINE deconvolution with TAM

library(BayesPrism); library(Seurat)

# 0) Paths
results_dir   <- "/content/drive/MyDrive/BayesPrism/results_annotation"
processed_dir <- "/content/drive/MyDrive/BayesPrism/processed_data"

# 1) Load the same single-cell counts and the refined labels (WITH TAM)
sc_matrix <- readRDS(file.path(results_dir, "cd45_counts.rds"))
refined_state <- readRDS(file.path(processed_dir, "refined_cell_types_final_WITH_TAM.rds"))
# Make sure names align to cells
names(refined_state) <- colnames(sc_matrix)

# 2) Define a *coarse* type for BayesPrism
coarse_from_state <- function(x){
  y <- character(length(x))
  y[grepl("^CD8\\+ T", x)]                 <- "CD8+ T"
  y[grepl("^CD4\\+ T", x)]                 <- "CD4+ T"
  y[grepl("^NK", x)]                       <- "NK cells"
  y[grepl("MAIT", x)]                      <- "MAIT cells"
  y[grepl("gamma delta", x, ignore.case=TRUE)] <- "γδ T"
  y[grepl("^B cells", x) | x=="B cells"]   <- "B cells"
  y[grepl("^mDC$|^pDC$|^Dendritic", x)]    <- "Dendritic cells"
  y[grepl("^Monocytes", x)]                <- "Monocytes"
  y[grepl("^TAM$|^Macrophage$", x)]        <- "Macrophage"
  # fallback: keep original if not matched
  y[y==""] <- x[y==""]
  y
}
refined_type <- coarse_from_state(refined_state)

# 3) Match genes and construct matrices
common_genes <- intersect(rownames(bulk_combined_with_symbols), rownames(sc_matrix))
# keep abundant sc genes to stabilize
top_genes <- names(sort(rowMeans(sc_matrix[common_genes, , drop=FALSE]), decreasing=TRUE))[1:min(18000, length(common_genes))]
common_genes <- intersect(common_genes, top_genes)

bulk_filtered <- bulk_combined_with_symbols[common_genes, , drop=FALSE]
sc_filtered   <- sc_matrix[common_genes, , drop=FALSE]

# Reference = cells x genes; mixture = samples x genes
sc_ref   <- t(sc_filtered)
bulk_mix <- t(bulk_filtered)

stopifnot(identical(colnames(sc_ref), colnames(bulk_mix)))
stopifnot(length(refined_state) == nrow(sc_ref))

cat("FINE reference states (n=", length(unique(refined_state)), "):\n", sep="")
print(sort(table(refined_state), decreasing=TRUE))

# 4) Build BayesPrism object (types + states)
bp_fine <- new.prism(
  reference = sc_ref,
  mixture = bulk_mix,
  input.type = "count",
  cell.type.labels  = refined_type,   # coarse types
  cell.state.labels = refined_state,  # fine states (with TAM)
  key = NULL,
  outlier.cut = 0.01,
  outlier.fraction = 0.05
)

# 5) Run deconvolution
bp_fine <- run.prism(
  bp_fine,
  n.cores = 8,
  update.gibbs = TRUE,
  gibbs.control = list(chain.length = 2000, burn.in = 500, thinning = 2)
)

# 6) Save results under distinct names
saveRDS(bp_fine, file.path(results_dir, "bayesprism_results_FINE_TAM.rds"))

# Export type/state fractions
theta_type  <- BayesPrism::get.fraction(bp_fine, which.theta="final", state.or.type="type")
theta_state <- BayesPrism::get.fraction(bp_fine, which.theta="final", state.or.type="state")

write.csv(t(theta_typ
            e),  file.path(results_dir, "theta_fractions_type_FINE_TAM.csv"))
write.csv(t(theta_state), file.path(results_dir, "theta_fractions_state_FINE_TAM.csv"))

cat("\nSaved:\n- bayesprism_results_FINE_TAM.rds\n- theta_fractions_type_FINE_TAM.csv\n- theta_fractions_state_FINE_TAM.csv\n")
